# XmR charts for non-manufacturing business metrics

**A runnable field guide for weekly business reviews: seasonality, new-location ramps, organic growth, and the other ways plain XmR charts mislead.**

Audience: a data product manager who is strong in SQL and functional in Python. Everything here is synthetic data with a fixed random seed and *known injected signals*, so after each chart you can check whether the method caught what was planted.

How to use this notebook
- Run it top to bottom (`Runtime → Run all`). The first cell installs the one package Colab does not ship with (`nbformat` is not needed; only `statsmodels` is checked).
- Section 0 defines one reusable `xmr()` function and a `plot_xmr()` helper. Every later section reuses them, so read Section 0 before skipping around.
- Each technique follows the same pattern: **the problem and the theory → the "wrong way" chart (false or missed signals) → the "right way" chart → deployment notes → further reading.**
- Links were checked against a search index on 2026-11-08. A link marked **(unverified)** could not be confirmed and is given for convenience only; the book/author citation beside it is the reliable reference.

Contents
0. Foundations: what XmR measures, why 2.66 and 3.268, detection rules, why never to chart rolling averages or cumulatives
1. Within-week seasonality: aggregate, stratify, or deseasonalize
2. Annual seasonality: Wheeler's seasonal factors vs. STL vs. year-over-year
3. New locations with no baseline: cohort ramp curves, provisional limits, graduation
4. Sustained growth: growth rates, index-relative charts, trending XmR, component decomposition
5. Limit recalculation and phase charts
6. Pain points common to other businesses (rates, rare events, bounded metrics, lagged outcomes, mix shift, roll-ups, calendar artifacts, targets)
7. Executive translation layer and the metric registry
8. Reading list

In [ ]:
# Colab ships numpy, pandas, matplotlib, scipy and statsmodels. This cell only installs
# statsmodels if it is missing (e.g. on a bare local venv). Nothing else is required.
import importlib, subprocess, sys
for pkg in ["statsmodels"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("dependencies ok")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20261108                      # fixed seed: every chart in this notebook is reproducible
rng = np.random.default_rng(SEED)    # one generator, used in order; re-run from the top to reproduce

# A small, fixed chart palette (colour-blind-checked categorical hues plus reserved
# "status" colours for the signal markers). Signals also differ by marker SHAPE so
# that identity never rests on colour alone.
PAL = dict(
    series="#2a78d6", series2="#eb6834", series3="#1baf7a", series4="#6250d6", series5="#eda100",
    cl="#0b0b0b", limit="#52514e", grid="#e1e0d9", muted="#898781", baseline="#cde2fb",
    r1="#d03b3b", r2="#6250d6", r3="#fab219", r4="#ec835a", target="#008300",
)
plt.rcParams.update({
    "figure.dpi": 100, "figure.figsize": (11, 5.2),
    "axes.grid": True, "grid.color": PAL["grid"], "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#898781", "ytick.color": "#898781",
    "legend.frameon": False, "font.size": 10,
})
pd.set_option("display.width", 140)
print("numpy", np.__version__, "| pandas", pd.__version__)

## 0. Foundations

### What an XmR chart measures
An XmR chart (also called an individuals-and-moving-range chart, I-MR chart, or *process behaviour chart*) answers one question about a sequence of numbers: **is this process behaving the same way it did before, or has something changed?** It does not tell you *why*; it tells you *whether* a value, or a pattern of values, is bigger than the routine wobble you should expect.

Walter Shewhart (1931) called the routine wobble **common-cause variation**: the sum of many small, unassignable influences that are always present. A change large enough to stand out from it is an **assignable (special) cause**: something specific happened, and it is worth the cost of going to find out what. The whole economics of the chart is about not chasing noise (false alarms waste investigation time and erode trust) while not missing real changes (missed signals waste the opportunity to act). Three-sigma limits are the compromise Shewhart found worked across a huge range of real processes, and Wheeler has argued repeatedly that the choice is empirical and economic, not derived from the normal distribution.

### The two statistics and why the constants are 2.66 and 3.268
For a sequence $x_1, \dots, x_n$ the chart uses only two numbers:

- the average, $\bar{x}$, which becomes the **central line (CL)**;
- the **average moving range**, $\overline{mR}$, the mean of $|x_i - x_{i-1}|$, which measures *short-term, point-to-point* variation.

The **natural process limits** are $\bar{x} \pm 2.66\,\overline{mR}$ and the upper range limit is $3.268\,\overline{mR}$. Where do those come from? For a subgroup of size $n=2$ the expected range of two values from a normal-ish process is $d_2 \sigma$ with $d_2 = 1.128$. So $\hat{\sigma} = \overline{mR} / 1.128$ and three sigma is $3/1.128 = 2.660$. Likewise the upper limit on a range of two values is $D_4\,\overline{mR}$ with $D_4 = 3.268$. Those constants are tabulated in every SPC text (NIST handbook; Wheeler & Chambers).

Why the moving range and not the ordinary standard deviation? Because the standard deviation of the whole series mixes *within-process* noise with *between-period* changes. If the level shifted halfway through, the global SD is inflated by the shift, the limits widen, and the shift hides itself. The moving range only compares neighbours, so it stays close to the true noise even when the level moves. Wheeler calls computing limits from the global SD "done wrong" and a chart built that way will routinely miss signals (Wheeler, *Individual Charts Done Right and Done Wrong*).

### Wheeler's principle: successive values must be logically comparable
The moving range is only a yardstick for routine variation if neighbouring points are the kind of thing you would expect to be alike. Wheeler's two requirements for an XmR chart to work (*What Makes the XmR Chart Work?*, 2012):

1. **Successive values must be logically comparable.** A Monday and a Saturday for a restaurant are not. A ramping new store's week 3 and week 4 are not. A metric with 28 days in it next to one with 31 is not.
2. **The moving ranges must capture local, short-term, routine variation.** If something systematic (seasonality, trend, mixed populations) sits between neighbours, the moving ranges are inflated by that systematic effect, the limits go wide, and the chart becomes insensitive.

Almost every technique in this notebook is a way of restoring logical comparability: aggregate, stratify, deseasonalize, index to a cohort age, detrend, normalise per selling day, or split the phases.

### Detection rules used here
Wheeler's four rules (as adopted in *Understanding Variation* and *Making Sense of Data*; the "Western Electric" zone tests are the same idea with slightly different counts):

| Rule | Pattern | What it is sensitive to |
|---|---|---|
| 1 | one point outside the natural process limits | a large, sudden change (or a one-off event) |
| 2 | 8 successive points on the same side of the central line | a small sustained shift |
| 3 | 3 of 4 successive points beyond 1 sigma on the same side | a moderate shift |
| 4 | 2 of 3 successive points beyond 2 sigma on the same side | a moderate-to-large shift |

Rule 1 alone is often enough and is the only rule with a negligible false-alarm rate. Each extra rule buys sensitivity at the cost of more false alarms; Wheeler recommends keeping Rules 2 to 4 "in your hip pocket" and switching them on deliberately (Wheeler & Stauffer, *When Should We Use Extra Detection Rules?*). `xmr()` below lets you choose.

### Why you must never chart rolling averages or month-to-date cumulatives
A 4-week rolling average of independent noise has lag-1 autocorrelation $(k-1)/k = 0.75$. Neighbouring points share three of four inputs, so (a) the moving ranges are tiny and the limits collapse inward, and (b) successive points drift together in long smooth runs. Both effects manufacture Rule 2 run signals out of pure noise. A month-to-date cumulative is worse: every point contains every previous point. The demonstration below counts the false signals.

### The reusable `xmr()` function and plotting helper
Everything after this cell reuses these. Design notes for a pandas reader who is new to SPC:
- `xmr()` returns an `XmR` object whose `.df` holds, per point: the value `x`, moving range `mr`, central line `cl`, limits `lnpl`/`unpl`, the sigma estimate, boolean columns `r1`..`r4` plus `mr_r1`, and `signal` (any rule fired).
- `baseline` controls which points are used to compute the central line and limits (all points by default; an integer means "the first n points"; a boolean mask or slice selects arbitrary points). The limits are then **extended** over every point so that later points are judged against the earlier baseline. That is how a chart becomes a prediction rather than a description.
- `rules` selects which detection rules to run; `run_length` lets you use 7 or 9 instead of 8.
- `plot_xmr()` draws the X panel and the mR panel, shades the baseline, and marks signals with distinct shapes and colours.

In [ ]:
XMR_X_CONST  = 2.66    # 3 / d2(n=2) = 3 / 1.128
XMR_MR_CONST = 3.268   # D4 for n=2


def _baseline_mask(baseline, n):
    """Turn the many ways of specifying a baseline into a boolean numpy mask of length n."""
    if baseline is None:
        return np.ones(n, dtype=bool)
    if isinstance(baseline, (int, np.integer)):
        m = np.zeros(n, dtype=bool); m[: int(baseline)] = True; return m
    if isinstance(baseline, slice):
        m = np.zeros(n, dtype=bool); m[baseline] = True; return m
    m = np.asarray(baseline, dtype=bool)
    assert len(m) == n, "baseline mask must have one entry per point"
    return m


def detect_signals(dev, sigma, rules=("r1", "r2", "r3", "r4"), run_length=8):
    """Apply Wheeler's detection rules to deviations from the central line.

    dev    : array of (x - cl), i.e. signed distance from the central line
    sigma  : the chart sigma (= mR-bar / 1.128); the limits sit at +/- 3 sigma
    Returns a dict of boolean arrays, one per rule, each True where that rule fires.
    """
    dev = np.asarray(dev, dtype=float)
    n = len(dev)
    out = {r: np.zeros(n, dtype=bool) for r in ("r1", "r2", "r3", "r4")}
    if n == 0 or not np.isfinite(sigma) or sigma <= 0:
        return out
    side = np.sign(dev)                                  # +1 above CL, -1 below, 0 exactly on it
    if "r1" in rules:                                    # Rule 1: beyond 3 sigma
        out["r1"] = np.abs(dev) > 3 * sigma
    if "r2" in rules:                                    # Rule 2: run of `run_length` on one side
        i = 0
        while i < n:
            if side[i] == 0:
                i += 1; continue
            j = i
            while j + 1 < n and side[j + 1] == side[i]:
                j += 1
            if j - i + 1 >= run_length:
                out["r2"][i : j + 1] = True
            i = j + 1
    if "r3" in rules:                                    # Rule 3: 3 of 4 beyond 1 sigma, same side
        for i in range(n - 3):
            w = dev[i : i + 4]
            for s in (1, -1):
                beyond = (s * w > 1 * sigma)
                if beyond.sum() >= 3:
                    out["r3"][i : i + 4] |= beyond
    if "r4" in rules:                                    # Rule 4: 2 of 3 beyond 2 sigma, same side
        for i in range(n - 2):
            w = dev[i : i + 3]
            for s in (1, -1):
                beyond = (s * w > 2 * sigma)
                if beyond.sum() >= 2:
                    out["r4"][i : i + 3] |= beyond
    return out


class XmR:
    """Container for one XmR chart: a DataFrame of per-point values plus summary stats."""
    def __init__(self, df, cl, mr_bar, sigma, name=""):
        self.df, self.cl, self.mr_bar, self.sigma, self.name = df, cl, mr_bar, sigma, name
        self.unpl, self.lnpl, self.url = cl + XMR_X_CONST * mr_bar, cl - XMR_X_CONST * mr_bar, XMR_MR_CONST * mr_bar

    def signals(self):
        """Tidy table of every point where at least one rule fired."""
        d = self.df
        rows = d[d["signal"]]
        rules = rows[["r1", "r2", "r3", "r4"]].apply(lambda r: ",".join(k for k, v in r.items() if v), axis=1)
        return pd.DataFrame({"x": rows["x"].round(3), "rules": rules, "side": np.where(rows["x"] > rows["cl"], "above", "below")})

    def n_signals(self):
        return {r: int(self.df[r].sum()) for r in ("r1", "r2", "r3", "r4", "mr_r1")}

    def __repr__(self):
        return (f"XmR({self.name!r}: n={len(self.df)}, CL={self.cl:.4g}, mR-bar={self.mr_bar:.4g}, "
                f"limits=[{self.lnpl:.4g}, {self.unpl:.4g}], signals={self.n_signals()})")


def xmr(values, index=None, baseline=None, rules=("r1", "r2", "r3", "r4"), run_length=8, name=""):
    """Build an XmR chart.

    values   : 1-D sequence of the metric, in time order (one value per period)
    index    : optional labels for the x-axis (dates, week numbers, cohort ages)
    baseline : which points define the central line and limits (None = all, int = first n,
               slice or boolean mask = those points). Limits are extended over all points.
    rules    : which of Wheeler's detection rules to run ("r1".."r4")
    """
    if index is None and isinstance(values, pd.Series):
        index = values.index                                   # keep dates/labels when a Series is passed
    x = pd.Series(np.asarray(values, dtype=float), index=(index if index is not None else pd.RangeIndex(len(values))))
    n = len(x)
    bmask = _baseline_mask(baseline, n)
    mr = x.diff().abs()
    # A moving range belongs to the baseline only if BOTH of its points are in the baseline.
    mr_in_base = bmask & np.roll(bmask, 1); mr_in_base[0] = False
    cl = float(x[bmask].mean())
    mr_bar = float(mr[mr_in_base].mean()) if mr_in_base.sum() > 0 else np.nan
    sigma = XMR_X_CONST * mr_bar / 3.0
    df = pd.DataFrame({"x": x, "mr": mr, "cl": cl, "unpl": cl + XMR_X_CONST * mr_bar, "lnpl": cl - XMR_X_CONST * mr_bar,
                       "sigma": sigma, "mr_cl": mr_bar, "mr_url": XMR_MR_CONST * mr_bar, "baseline": bmask})
    flags = detect_signals(df["x"].values - cl, sigma, rules, run_length)
    for k, v in flags.items():
        df[k] = v
    df["mr_r1"] = (df["mr"] > df["mr_url"]).fillna(False).astype(bool)
    df["signal"] = df[["r1", "r2", "r3", "r4"]].any(axis=1)
    return XmR(df, cl, mr_bar, sigma, name)


def xmr_fixed(values, index=None, cl=None, mr_bar=None, name="", rules=("r1", "r2", "r3", "r4"), run_length=8):
    """XmR chart with a central line and mR-bar supplied from OUTSIDE the data (e.g. borrowed from
    prior cohorts, or frozen limits from an earlier phase). Detection rules run against those limits."""
    if index is None and isinstance(values, pd.Series):
        index = values.index
    x = pd.Series(np.asarray(values, dtype=float), index=(index if index is not None else pd.RangeIndex(len(values))))
    sigma = XMR_X_CONST * mr_bar / 3.0
    df = pd.DataFrame({"x": x, "mr": x.diff().abs(), "cl": cl, "unpl": cl + XMR_X_CONST * mr_bar, "lnpl": cl - XMR_X_CONST * mr_bar,
                       "sigma": sigma, "mr_cl": mr_bar, "mr_url": XMR_MR_CONST * mr_bar, "baseline": np.zeros(len(x), dtype=bool)})
    flags = detect_signals(df["x"].values - cl, sigma, rules, run_length)
    for k, v in flags.items():
        df[k] = v
    df["mr_r1"] = (df["mr"] > df["mr_url"]).fillna(False).astype(bool)
    df["signal"] = df[["r1", "r2", "r3", "r4"]].any(axis=1)
    return XmR(df, cl, mr_bar, sigma, name)


SIGNAL_STYLE = {  # rule -> (marker, colour, label); shapes differ so colour is never the only cue
    "r1": ("X", PAL["r1"], "Rule 1: outside limits"),
    "r2": ("s", PAL["r2"], "Rule 2: 8+ run on one side"),
    "r3": ("^", PAL["r3"], "Rule 3: 3 of 4 beyond 1σ"),
    "r4": ("D", PAL["r4"], "Rule 4: 2 of 3 beyond 2σ"),
}


def plot_xmr(res, title="", units="", show_mr=True, ax=None, target=None, target_label="goal",
             annotate=None, ylim=None, step_limits=False, figsize=None):
    """Draw an XmR chart. `res` is an XmR object. Returns the matplotlib figure.

    annotate : optional dict {x_position: "text"} to mark injected events on the chart
    target   : optional horizontal goal line (drawn in a distinct colour, never confused with limits)
    step_limits : draw cl/limits as steps (for phase charts whose limits change)
    """
    d = res.df
    xs = d.index
    if ax is None:
        if show_mr:
            fig, (ax, axm) = plt.subplots(2, 1, sharex=True, figsize=figsize or (11, 6.2),
                                          gridspec_kw={"height_ratios": [2.2, 1], "hspace": 0.08})
        else:
            fig, ax = plt.subplots(figsize=figsize or (11, 4.2)); axm = None
    else:
        fig, axm = ax.figure, None
    ds = "steps-post" if step_limits else "default"
    # baseline shading: only when the baseline is a strict subset of the points
    if d["baseline"].any() and not d["baseline"].all():
        lo_y = ylim[0] if (ylim and ylim[0] is not None) else d[["x", "lnpl"]].min().min()
        hi_y = ylim[1] if (ylim and ylim[1] is not None) else d[["x", "unpl"]].max().max()
        ax.fill_between(xs, lo_y, hi_y, where=d["baseline"].values, color=PAL["baseline"], alpha=0.35, linewidth=0,
                        label="baseline (limits computed here)")
    ax.plot(xs, d["x"], color=PAL["series"], lw=1.4, marker="o", ms=3.2, label=res.name or "value")
    ax.plot(xs, d["cl"], color=PAL["cl"], lw=1.2, drawstyle=ds, label="central line")
    ax.plot(xs, d["unpl"], color=PAL["limit"], lw=1.0, ls="--", drawstyle=ds, label="natural process limits")
    ax.plot(xs, d["lnpl"], color=PAL["limit"], lw=1.0, ls="--", drawstyle=ds)
    for r, (mk, col, lab) in SIGNAL_STYLE.items():
        m = d[r].values
        if m.any():
            ax.plot(xs[m], d["x"].values[m], ls="none", marker=mk, ms=8 if r == "r1" else 6.5, color=col,
                    markeredgecolor="white", markeredgewidth=0.6, label=lab, zorder=5)
    if target is not None:
        ax.axhline(target, color=PAL["target"], lw=1.4, ls=(0, (6, 3)), label=target_label)
    if annotate:
        y0, y1 = ax.get_ylim()
        for i, (xpos, txt) in enumerate(annotate.items()):
            ax.axvline(xpos, color=PAL["muted"], lw=0.8, ls=":")
            ax.text(xpos, y1 - (i % 3) * 0.09 * (y1 - y0), " " + txt, color=PAL["muted"], fontsize=8.5, va="top", ha="left")
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_title(title, loc="left", fontsize=12, color=PAL["cl"])
    ax.set_ylabel(units)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8.5)
    if axm is not None:
        axm.plot(xs, d["mr"], color=PAL["series2"], lw=1.1, marker="o", ms=2.6, label="moving range")
        axm.plot(xs, d["mr_cl"], color=PAL["cl"], lw=1.0, drawstyle=ds)
        axm.plot(xs, d["mr_url"], color=PAL["limit"], lw=1.0, ls="--", drawstyle=ds, label="upper range limit")
        m = d["mr_r1"].values
        if m.any():
            axm.plot(xs[m], d["mr"].values[m], ls="none", marker="X", ms=7, color=PAL["r1"], label="mR beyond URL")
        axm.set_ylabel("mR")
        axm.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8.5)
        if isinstance(xs, pd.DatetimeIndex):
            axm.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    elif isinstance(xs, pd.DatetimeIndex):
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    return fig


def report(res, injected=None):
    """Print the signal summary and, if you tell it where the signal was injected, whether it was caught."""
    print(res)
    s = res.signals()
    print(f"  signal points: {len(s)}")
    if injected is not None:
        pos = res.df.index.get_indexer([injected])[0] if injected in res.df.index else None
        caught = bool(res.df["signal"].iloc[pos:].any()) if pos is not None else None
        print(f"  injected change at {injected}: {'CAUGHT ✅' if caught else 'MISSED ❌'}")
    return s


print("xmr(), plot_xmr(), report() defined")

#### Sanity check on a stable series with one injected shift
Forty weeks of stable noise, then the level steps up by 1.5 standard deviations at week 41. Rule 1 may or may not fire on a 1.5-sigma shift (it usually does not); Rules 2 to 4 are what catch moderate shifts. Look also at the handful of Rule 2–4 flags that appear *inside* the shaded baseline, before anything was injected: that is the false-alarm cost of the extra rules on ordinary noise, and it is why you switch them on deliberately rather than by default.

In [ ]:
weeks = pd.date_range("2024-01-07", periods=60, freq="W-SUN")
base = 500 + rng.normal(0, 20, size=60)
shift_at = weeks[40]
y = base + np.where(weeks >= shift_at, 30, 0)         # +30 = 1.5 sigma injected level shift from week 41

res0 = xmr(y, index=weeks, baseline=40, name="weekly bookings")   # limits from the first 40 weeks, extended forward
plot_xmr(res0, "0.1 A stable process with a +1.5σ level shift injected at week 41 (limits from first 40 weeks)",
         units="bookings / week", annotate={shift_at: "injected +30"})
report(res0, injected=shift_at)

#### The rolling-average trap, quantified
Pure independent noise, charted three ways: the raw values, a 4-week rolling average, and a month-to-date cumulative. The raw chart should produce almost no signals. Watch what the other two do.

In [ ]:
n = 104
noise = pd.Series(100 + rng.normal(0, 10, size=n), index=pd.date_range("2024-01-07", periods=n, freq="W-SUN"))

raw  = xmr(noise, name="raw weekly values")
roll = xmr(noise.rolling(4).mean().dropna(), name="4-week rolling average")
# running cumulative (year-to-date style): every point contains every previous point
cum  = xmr(noise.cumsum(), name="running cumulative (YTD)")

for r in (raw, roll, cum):
    lag1 = pd.Series(r.df["x"].values).autocorr(lag=1)
    print(f"{r.name:28s} lag-1 autocorrelation = {lag1:5.2f}   signals by rule: {r.n_signals()}")

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=False)
for ax, r, t in zip(axes, (raw, roll, cum), ("0.2a Raw independent noise: a healthy, boring chart",
                                             "0.2b Same noise, 4-week rolling average: smooth runs that are pure artefact",
                                             "0.2c Same noise, running cumulative: a monotone \"trend\" built from nothing")):
    plot_xmr(r, t, ax=ax, show_mr=False)
plt.tight_layout()

**Reading the result.** The raw chart has about the number of flagged points you would expect from four rules on 104 noise points (Rules 3 and 4 each carry a non-trivial false-alarm rate; this is why Wheeler keeps them optional). The rolling average has lag-1 autocorrelation near 0.75, its limits are far too tight (the moving ranges of a smoothed series are small), and the run rule fires on long stretches that are nothing but shared inputs. The running cumulative is a monotone "trend" assembled from nothing, so Rule 2 fires on essentially every point. A month-to-date cumulative that resets on the 1st fails the opposite way: the reset produces huge moving ranges, the limits balloon, and the chart can no longer see anything.

**Deployment rule:** chart the *period value*, never a smoothed or accumulated version of it. If the period value is too noisy to be useful, change the grain (daily → weekly) rather than smoothing: aggregation of non-overlapping windows keeps points independent; rolling windows do not.

**Further reading (Section 0)**
- Wheeler, D. J. *Understanding Variation: The Key to Managing Chaos*, 2nd ed., SPC Press, 2000. The 150-page introduction; chapters 2–4 cover the chart, the limits, and the detection rules.
- Wheeler, D. J. & Chambers, D. S. *Understanding Statistical Process Control*, 3rd ed., SPC Press, 2010. Ch. 9.2 "Individual and Moving Range Charts Done Right"; tables of d2 and D4.
- Wheeler, "What Makes the XmR Chart Work?", Quality Digest, Nov 2012: https://www.qualitydigest.com/node/22342/ (the "logically comparable" principle)
- Wheeler, "Individual Charts Done Right and Done Wrong", Quality Digest, Feb 2010: https://www.qualitydigest.com/node/10006/
- Wheeler, "Why Three-Sigma Limits?", Quality Digest (1996): https://www.qualitydigest.com/aug/wheeler.html
- Wheeler, "Foundations of Shewhart's Charts", Quality Digest, Oct 1996: https://www.qualitydigest.com/oct96/spctool.html
- Wheeler & Stauffer, "When Should We Use Extra Detection Rules?", Quality Digest: https://www.qualitydigest.com/node/30664
- Wheeler, "Autocorrelated Data", Quality Digest, Aug 2017: https://www.qualitydigest.com/inside/statistics-column/autocorrelated-data-080717.html
- NIST/SEMATECH *e-Handbook of Statistical Methods*, §6.3.2.2 Individuals Control Charts: https://www.itl.nist.gov/div898/handbook/pmc/section3/pmc322.htm (unverified; the handbook index at https://www.itl.nist.gov/div898/handbook/pmc/pmc.htm is confirmed)
- Western Electric zone rules (the 4-of-5 variant of Rule 3): https://en.wikipedia.org/wiki/Western_Electric_rules
- Shewhart, W. A. *Economic Control of Quality of Manufactured Product*, Van Nostrand, 1931 (the origin of three-sigma limits and the common/assignable cause distinction).

## 1. Within-week seasonality (day-of-week patterns)

### The problem
Most service businesses have a weekly rhythm: a dental chain is quiet on Mondays, a restaurant peaks Saturday, a B2B SaaS funnel dies at the weekend. Put daily values on one XmR chart and you break Wheeler's first principle: a Monday and a Saturday are not logically comparable. Two things go wrong at once.

1. **The moving ranges are inflated by the day-of-week swing**, not by noise. The limits widen to accommodate the pattern. A real change in the underlying level has to be enormous before it crosses them, so the chart becomes insensitive. Worse, it becomes insensitive *asymmetrically*: peak days sit near the upper limit and trough days near the lower one, so a rise is only ever noticed on Saturdays and a fall only on Mondays.
2. **Rules 3 and 4 can fire on the pattern itself** (consecutive peak days all above 1 sigma), producing false alarms that teach the reviewers to ignore the chart.

### Three remedies, in order of preference
1. **Aggregate to the weekly grain.** The week is the natural unit; summing 7 days removes the within-week pattern exactly and also averages away some noise. Cost: 7× fewer points, so detection waits for the week to close. This is the right default for a weekly business review.
2. **Stratify: one chart per weekday.** Keeps the daily grain and is the best choice when the question is itself day-specific ("has Saturday changed?"). Cost: 7 charts, each with 1/7 of the points; a change that affects all days is spread across all seven charts and looks weaker on each.
3. **Deseasonalize with day-of-week factors.** Compute each weekday's average as a fraction of the grand average over a *signal-free baseline*, divide each day's value by its factor, chart the result. This is Wheeler's seasonal-factor method applied with a 7-period cycle. You keep daily points and a single chart. Cost: the factors are estimates and must be governed (see deployment notes).

Below: 26 weeks of daily volume with a strong day-of-week pattern and a **−12% level drop injected at the start of week 19**. The −12% is chosen because it is large enough to matter commercially and small enough that a mis-specified chart misses it.

In [ ]:
def small_multiples(series_dict, baseline=None, ncols=2, suptitle="", annotate=None, rules=("r1", "r2", "r3", "r4")):
    """One small XmR panel per series (dict name -> Series). Returns {name: XmR}."""
    n = len(series_dict); nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(11, 2.5 * nrows + 0.6), squeeze=False)
    results = {}
    for ax, (k, s) in zip(axes.flat, series_dict.items()):
        r = xmr(s, baseline=baseline, rules=rules, name=k); results[k] = r
        plot_xmr(r, k, ax=ax, show_mr=False)
        ax.get_legend().remove(); ax.tick_params(labelsize=8)
        if annotate:
            for xpos in annotate:
                ax.axvline(xpos, color=PAL["muted"], lw=0.8, ls=":")
    for ax in list(axes.flat)[n:]:
        ax.set_visible(False)
    handles = [Line2D([], [], marker=mk, color=col, ls="none", label=lab) for mk, col, lab in SIGNAL_STYLE.values()]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8.5, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(suptitle, x=0.01, ha="left", fontsize=12)
    fig.tight_layout(rect=(0, 0.03, 1, 0.97))
    return results


# --- synthetic daily data with a day-of-week pattern ---------------------------------
days = pd.date_range("2025-01-06", periods=26 * 7, freq="D")          # starts on a Monday
DOW_TRUE = np.array([0.80, 0.85, 0.90, 0.95, 1.15, 1.40, 0.95])       # Mon..Sun multiplicative pattern
level_daily = 400
drop_day = days[18 * 7]                                               # first day of week 19
true_daily = level_daily * DOW_TRUE[days.dayofweek] * np.where(days >= drop_day, 0.88, 1.0)
daily = pd.Series(np.round(true_daily * rng.lognormal(0, 0.06, len(days))), index=days, name="visits")
print(daily.head(8))

#### Wrong way: daily values on one chart

In [ ]:
res_wrong = xmr(daily, baseline=18 * 7, name="daily visits (raw)")
plot_xmr(res_wrong, "1.1 Wrong: daily visits on one chart. Limits are set by the weekly rhythm, not by noise",
         units="visits / day", annotate={drop_day: "−12% injected"})
report(res_wrong, injected=drop_day)
sig = res_wrong.signals()
pre, post = sig[sig.index < drop_day], sig[sig.index >= drop_day]
print(f"flags BEFORE the drop (all false alarms): {len(pre)} on", sorted(set(pd.DatetimeIndex(pre.index).day_name())))
print(f"flags AFTER the drop: {len(post)}, rules used:", sorted(set(post['rules'])), "on", sorted(set(pd.DatetimeIndex(post.index).day_name())))
res_wrong_r1 = xmr(daily, baseline=18 * 7, rules=("r1", "r2"), name="daily visits (raw), Rules 1-2 only")
print("with Wheeler's default (Rule 1 plus the run rule) only:",
      "CAUGHT" if res_wrong_r1.df['signal'][res_wrong_r1.df.index >= drop_day].any() else "MISSED ❌")
print(f"sigma estimated by this chart: {res_wrong.sigma:.1f} visits/day; true daily noise is about {0.06 * level_daily:.0f}")

Read the two failure modes off this chart. **Before** the drop, Rule 1 fires on ordinary Saturdays: false alarms, because the limits describe a mixture of weekdays and the busiest weekday lives next to the upper limit. **After** the drop, the only evidence is the weak zone test (Rule 3) on the trough days; with Wheeler's default rule set (Rule 1 plus the run rule) the −12% is missed entirely, because sigma has been inflated almost threefold by the weekly rhythm. The chart has no way to say "every weekday is down 12%".

#### Right way (a): aggregate to weeks

In [ ]:
weekly = daily.resample("W-SUN").sum()                 # complete weeks because the series starts on a Monday
res_week = xmr(weekly, baseline=18, name="weekly visits")
plot_xmr(res_week, "1.2 Right (a): aggregate to the weekly grain; the −12% week is a clear Rule 1 signal",
         units="visits / week", annotate={weekly.index[18]: "−12% injected"})
report(res_week, injected=weekly.index[18])

#### Right way (b): stratify, one chart per weekday

In [ ]:
by_dow = {d: daily[daily.index.day_name() == d] for d in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]}
res_dow = small_multiples(by_dow, baseline=18, ncols=2, suptitle="1.3 Right (b): one chart per weekday (limits from first 18 weeks)",
                          annotate=[drop_day])
for k, r in res_dow.items():
    first = r.df.index[(r.df.index >= drop_day) & r.df["signal"]]
    print(f"{k:10s} first signal after the drop: {first[0].date() if len(first) else 'none'}")

#### Right way (c): deseasonalize with day-of-week factors
The factors come from the baseline only (weeks 1–18). Dividing by them makes Monday and Saturday comparable, so the daily grain survives.

In [ ]:
def seasonal_factors(series, period_key, baseline_mask=None, mode="multiplicative"):
    """Wheeler-style seasonal factors.

    series        : pd.Series of the metric
    period_key    : array-like of the season position for each point (e.g. weekday 0..6, week-of-year 0..51)
    baseline_mask : which points to learn the factors from (should be a signal-free period). Default: all.
    mode          : "multiplicative" (volumes: factor = period mean / grand mean) or
                    "additive" (rates: offset = period mean - grand mean)
    Returns a pd.Series of factors indexed by season position.
    """
    key = pd.Series(np.asarray(period_key), index=series.index)
    m = np.ones(len(series), dtype=bool) if baseline_mask is None else np.asarray(baseline_mask, dtype=bool)
    grand = series[m].mean()
    per = series[m].groupby(key[m]).mean()
    return (per / grand) if mode == "multiplicative" else (per - grand)


def deseasonalize(series, period_key, factors, mode="multiplicative"):
    f = factors.reindex(np.asarray(period_key)).values
    return (series / f) if mode == "multiplicative" else (series - f)


base_mask = daily.index < drop_day
dow_factors = seasonal_factors(daily, daily.index.dayofweek, baseline_mask=base_mask)
print("estimated day-of-week factors (Mon..Sun):", dow_factors.round(3).tolist())
print("true factors used to generate the data   :", (DOW_TRUE / DOW_TRUE.mean()).round(3).tolist())

adj = deseasonalize(daily, daily.index.dayofweek, dow_factors)
res_adj = xmr(adj, baseline=18 * 7, name="daily visits ÷ DOW factor")
plot_xmr(res_adj, "1.4 Right (c): daily values divided by their day-of-week factor, one chart, daily grain",
         units="deseasonalized visits / day", annotate={drop_day: "−12% injected"})
report(res_adj, injected=drop_day)
first = res_adj.df.index[(res_adj.df.index >= drop_day) & res_adj.df["signal"]]
print("first signal after the drop:", first[0].date() if len(first) else "none",
      f"({(first[0]-drop_day).days} days after it started)" if len(first) else "")

# --- scorecard: false alarms in the baseline vs. detection delay, for each approach ---
def scorecard(res, change_at):
    d = res.df
    before = d[(d.index < change_at)]["signal"].sum()
    after = d.index[(d.index >= change_at) & d["signal"]]
    delay = (after[0] - change_at).days if len(after) else None
    return dict(approach=res.name, sigma_pct_of_cl=round(100 * res.sigma / res.cl, 1), false_flags_in_baseline=int(before),
                days_to_first_signal=delay)

rows = [scorecard(res_wrong, drop_day), scorecard(res_week, weekly.index[18]), scorecard(res_adj, drop_day)]
rows += [scorecard(r, drop_day) for r in res_dow.values()]
print("sigma as % of the central line is the sensitivity of each chart; the raw daily chart is 3x less sensitive than the fixes")
pd.DataFrame(rows)

### Deployment notes (within-week seasonality)
- **Grain.** For a weekly business review, aggregate to weeks and chart the weekly value; keep the deseasonalized daily chart for operators who need to react mid-week. Never chart a 7-day rolling sum (Section 0).
- **Factors.** Learn day-of-week factors from a signal-free window of at least 8–12 weeks; store them with the metric (Section 7's registry). Recompute them only when you re-baseline the chart, not every week: factors that move every week silently absorb real changes.
- **Partial weeks.** The current, incomplete week must never appear on the weekly chart. In SQL, filter `week_end < current_date`.
- **Holidays.** A public holiday on a Tuesday is a different "season" from a normal Tuesday. Either exclude holiday days from the daily chart (and annotate the gap) or give holidays their own factor if you have enough of them. On the weekly chart, annotate holiday weeks and treat a Rule 1 point on one as a known cause, not a mystery.
- **Governance.** Each chart should state in its subtitle which weeks the limits came from. If the review cannot see that, nobody can challenge it.

**Further reading (Section 1)**
- Stacey Barr, "CASE STUDY: 3 Common Mistakes to Avoid With Your XmR Charts (Part 1)" (mistake 1 is seasonality; the deseasonalising method is credited to Wheeler): https://staceybarr.com/?p=2759
- Wheeler, "What Makes the XmR Chart Work?" (logical comparability), Quality Digest, 2012: https://www.qualitydigest.com/node/22342/
- Wheeler, "Rational Subgrouping", Quality Digest, June 2015 (why how you group the data decides what the chart can answer): https://www.qualitydigest.com/node/26952/
- Wheeler, "What Is a Rational Subgroup?", Quality Digest, Oct 1997: https://www.qualitydigest.com/oct97/html/spctool.html
- Wheeler, D. J. *Making Sense of Data*, SPC Press, 2003: the chapters on seasonal data give the period-average ÷ grand-average method used here.

## 2. Annual seasonality

### The problem
With a yearly cycle the within-week fix (aggregate up) is not available: nobody runs a business review on annual totals. Weekly data with a ±20% annual swing fails both of Wheeler's requirements: a week in June is not comparable to one in December, and the moving ranges contain the seasonal slope. The raw chart fires false Rule 1 signals at every peak and trough, fires Rule 2 runs across every half-cycle, and is numb to a genuine level change that happens during a seasonal rise.

### Wheeler's seasonal-factor method
For each season position $k$ (week-of-year, month, fiscal period):
$$\text{factor}_k = \frac{\text{average of all baseline values at position } k}{\text{grand average of the baseline}}$$
and the deseasonalized value is actual ÷ factor. The factors must come from **whole cycles** (one year gives one observation per factor; two or three are much better) and from **signal-free periods**: a one-off outage in week 18 of year 2 will otherwise be baked into factor 18 and every future week 18 will look "up". The method is multiplicative for **volumes** (the seasonal swing scales with the level), additive for **rates and percentages** (the swing is a fixed number of points; a multiplicative factor on a 97% uptime figure makes no sense). With 52 weekly positions and only two or three years, individual factors are noisy; a short centred moving average *of the factors* (circular, 3–5 weeks) is a legitimate smoother because it is applied to the factor table, not to the series being charted.

### Comparing against two popular alternatives
- **STL decomposition** (Cleveland et al. 1990, `statsmodels.tsa.seasonal.STL`) fits a loess seasonal component and a loess trend. Charting the STL **remainder** sounds attractive but has a trap: the trend component is flexible and will *absorb a level shift within a trend-window of it*, leaving only a transient in the remainder. The fix is to use STL as a factor estimator on a frozen baseline and extend its seasonal component forward, rather than letting it re-fit through the data you are judging.
- **Year-over-year % change** is the business default. It removes seasonality exactly but has lag-1 noise amplified by √2 (two noisy numbers), has a 52-week lag before it exists at all, and, crucially, **echoes last year's anomaly as this year's signal**: an outage in week 18 of year 2 makes week 18 of year 3 look +50%.

Data below: 156 weeks (three years) of weekly volume, multiplicative annual cycle with a December spike, 4% noise, a **one-off −35% outage in week 70** (year 2) and a **+12% permanent level shift from week 120** (year 3).

In [ ]:
W = 156
t = np.arange(W)
wk = pd.date_range("2023-01-01", periods=W, freq="W-SUN")
pos = t % 52                                                   # season position 0..51 (synthetic: no 53-week years)
season_true = 1 + 0.22 * np.sin(2 * np.pi * (pos - 8) / 52) + 0.15 * np.exp(-0.5 * ((pos - 50) / 1.5) ** 2)
LEVEL = 1000
SHIFT_WK, ONEOFF_WK = 120, 70
true_w = LEVEL * season_true * np.where(t >= SHIFT_WK, 1.12, 1.0) * np.where(t == ONEOFF_WK, 0.65, 1.0)
vol = pd.Series(np.round(true_w * rng.lognormal(0, 0.04, W)), index=wk, name="orders")
ann = {wk[ONEOFF_WK]: "one-off −35%", wk[SHIFT_WK]: "+12% level shift"}

res_raw = xmr(vol, baseline=104, name="weekly orders (raw)")
plot_xmr(res_raw, "2.1 Wrong: raw weekly orders. Every peak and trough is a 'signal'; the real +12% shift is lost in the seasonal rise",
         units="orders / week", annotate=ann)
report(res_raw, injected=wk[SHIFT_WK])
print("false Rule 1 points inside the two baseline years:", int(res_raw.df["r1"].iloc[:104].sum() - 1), "(excluding the real one-off)")

#### Right way: Wheeler's seasonal factors, learned from signal-free whole cycles

In [ ]:
def smooth_circular(factors, window=5):
    """Centred moving average of a seasonal-factor table, wrapping around the year end."""
    f = factors.values; k = window // 2
    ext = np.concatenate([f[-k:], f, f[:k]])
    sm = np.convolve(ext, np.ones(window) / window, mode="valid")
    return pd.Series(sm / sm.mean(), index=factors.index)     # renormalise so the factors average 1


baseline_years = t < 104                                        # first two full cycles
# (1) naive: learn factors from everything in the baseline, including the week-70 outage
f_naive = seasonal_factors(vol, pos, baseline_mask=baseline_years)
# (2) governed: exclude known signal weeks (the outage was a Rule 1 point with a known cause) before learning factors
clean_mask = baseline_years & (t != ONEOFF_WK)
f_clean = smooth_circular(seasonal_factors(vol, pos, baseline_mask=clean_mask), window=5)

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(f_naive.index, f_naive.values, color=PAL["series2"], lw=1.2, label="factors incl. outage week (naive)")
ax.plot(f_clean.index, f_clean.values, color=PAL["series"], lw=2, label="factors excl. outage, 5-wk circular smooth")
ax.plot(np.arange(52), season_true[:52] / season_true[:52].mean(), color=PAL["cl"], lw=1, ls=":", label="true seasonal pattern")
ax.set_title("2.2 Seasonal factors: position 18 is corrupted by the one-off unless you exclude it", loc="left")
ax.set_xlabel("week of year"); ax.set_ylabel("factor"); ax.legend(fontsize=8.5)
plt.show()

adj_vol = deseasonalize(vol, pos, f_clean)
res_adj2 = xmr(adj_vol, baseline=104, name="orders ÷ seasonal factor")
plot_xmr(res_adj2, "2.3 Right: deseasonalized orders. The outage is a one-off Rule 1; the +12% shift is caught by Rules 1/4 then held by Rule 2",
         units="deseasonalized orders / week", annotate=ann)
report(res_adj2, injected=wk[SHIFT_WK])
first = res_adj2.df.index[(t >= SHIFT_WK) & res_adj2.df["signal"].values]
print("weeks from shift to first signal:", (first[0] - wk[SHIFT_WK]).days // 7 if len(first) else "n/a")

#### Additive factors for a rate
Same idea for a percentage: subtract the seasonal *offset* instead of dividing. A conversion rate that runs 1.5 points higher in Q4 needs an additive correction; dividing by a factor would make the correction scale with the level, which is wrong for a bounded rate.

In [ ]:
conv_true = 0.050 + 0.015 * np.sin(2 * np.pi * (pos - 30) / 52) + np.where(t >= 130, 0.008, 0)   # +0.8 pt shift at week 130
conv = pd.Series(conv_true + rng.normal(0, 0.004, W), index=wk, name="conversion rate")
off = seasonal_factors(conv, pos, baseline_mask=baseline_years, mode="additive")
off = smooth_circular(off + 1, 5) - 1                            # smooth (shift by 1 so the normaliser is harmless), then shift back
conv_adj = deseasonalize(conv, pos, off, mode="additive")
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5))
plot_xmr(xmr(conv, baseline=104, name="raw rate"), "2.4a Raw conversion rate: seasonal swing drives false signals", ax=axes[0], show_mr=False)
plot_xmr(xmr(conv_adj, baseline=104, name="rate − seasonal offset"), "2.4b Additive deseasonalization: +0.8 pt shift at week 130 is visible", ax=axes[1], show_mr=False)
for a in axes: a.axvline(wk[130], color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()

#### Alternative 1: STL remainder (statsmodels)

In [ ]:
from statsmodels.tsa.seasonal import STL

# A fourth year is added for this demonstration so there is enough post-shift data to watch the STL trend adapt.
t4 = np.arange(208); pos4 = t4 % 52
season4 = 1 + 0.22 * np.sin(2 * np.pi * (pos4 - 8) / 52) + 0.15 * np.exp(-0.5 * ((pos4 - 50) / 1.5) ** 2)
extra = np.round(LEVEL * season4[156:] * 1.12 * rng.lognormal(0, 0.04, 52))
vol4 = pd.concat([vol, pd.Series(extra, index=pd.date_range(wk[-1] + pd.Timedelta(days=7), periods=52, freq="W-SUN"))])
wk4 = vol4.index

# (a) The naive use: fit STL through ALL the data and chart the remainder.
stl_all = STL(np.log(vol4), period=52, seasonal=13).fit()
rem_all = pd.Series(np.exp(stl_all.resid.values), index=wk4)      # ratio scale: 1.0 = "as the decomposition expected"
res_stl_all = xmr(rem_all, baseline=104, name="STL remainder (fit on all data)")

# (b) The governed use: fit STL on the frozen baseline only, take its seasonal component,
#     extend it forward by period, and chart actual ÷ seasonal. No trend is fitted through the data under review.
stl_base = STL(np.log(vol4.iloc[:104]), period=52, seasonal=13).fit()
seas_52 = pd.Series(stl_base.seasonal.values, index=pos4[:104]).groupby(level=0).mean()   # mean seasonal effect per position
seas_ext = np.exp(seas_52.reindex(pos4).values)
res_stl_base = xmr(vol4 / seas_ext, baseline=104, name="orders ÷ STL seasonal (baseline-frozen)")

fig, axes = plt.subplots(3, 1, figsize=(11, 10))
axes[0].plot(wk4, np.exp(stl_all.trend.values), color=PAL["series3"], lw=1.6, label="STL trend (fit on all data)")
axes[0].plot(wk4, LEVEL * np.where(t4 >= SHIFT_WK, 1.12, 1.0), color=PAL["cl"], ls=":", lw=1, label="true level")
axes[0].set_title("2.5a The loess trend bends through the +12% step and has fully absorbed it within about a year", loc="left"); axes[0].legend(fontsize=8.5)
plot_xmr(res_stl_all, "2.5b Wrong: charting the STL remainder; the shift appears briefly, then 'expires' as the trend catches up", ax=axes[1], show_mr=False)
plot_xmr(res_stl_base, "2.5c Right: STL used only to estimate seasonal factors on a frozen baseline; the shift persists", ax=axes[2], show_mr=False)
for a in axes[1:]:
    for xpos in ann: a.axvline(xpos, color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()
for r in (res_stl_all, res_stl_base):
    post = r.df["signal"].values
    print(f"{r.name:45s} flagged points: weeks 120-139: {int(post[120:140].sum()):2d}/20   weeks 160-207: {int(post[160:].sum()):2d}/48")

#### Alternative 2: year-over-year % change

In [ ]:
yoy = (vol / vol.shift(52) - 1).dropna() * 100
res_yoy = xmr(yoy, baseline=52, name="YoY % change")
plot_xmr(res_yoy, "2.6 YoY % change: the week-70 outage reappears 52 weeks later as a fake +50% 'signal'",
         units="% vs same week last year",
         annotate={wk[ONEOFF_WK]: "real outage", wk[ONEOFF_WK + 52]: "echo (false)", wk[SHIFT_WK]: "+12% shift"})
report(res_yoy)
print("YoY value in the echo week:", round(float(yoy.loc[wk[ONEOFF_WK + 52]]), 1), "%")
print("noise sd of YoY vs deseasonalized ratio:", round(float(yoy.iloc[:52].std()), 2), "pts vs",
      round(float((adj_vol.iloc[:104] / adj_vol.iloc[:104].mean() * 100).std()), 2), "pts")

### Moving holidays and factor governance
- **Moving holidays** (Easter, Thanksgiving week, Ramadan, Lunar New Year, Black Friday if your fiscal weeks shift) do not sit at a fixed week-of-year position. Two options: (1) define the season position on a *holiday-aligned calendar* (e.g. "week containing Easter" is its own position, and the surrounding weeks are indexed relative to it); or (2) treat holiday weeks as known assignable causes: exclude them from factor estimation, annotate them on the chart, and expect a Rule 1 point you do not investigate. Option 2 is simpler and is usually enough for a weekly review. Retail-calendar businesses should use their 4-4-5 fiscal weeks as the position key so that positions align year to year.
- **53-week years.** In an ISO calendar some years have 53 weeks; map week 53 onto position 52's factor (or average 51 and 52) and document it.
- **Recompute factors only from signal-free periods**, only when you re-baseline (Section 5), and keep the previous factor table so a reviewer can reproduce old charts. A factor table that silently changes is a chart that cannot be audited.
- **Pre-register the method per metric**: multiplicative or additive, how many cycles, smoothing window. That is the "seasonality model" field of the metric registry in Section 7.

**Further reading (Section 2)**
- Wheeler, D. J. *Making Sense of Data*, SPC Press, 2003 (seasonal-factor method; the worked examples use monthly data).
- Stacey Barr, "CASE STUDY: 3 Common Mistakes to Avoid With Your XmR Charts (Part 1)": https://staceybarr.com/?p=2759 and her Smart Charts Q&A note on recomputing factors from the latest cycles: https://staceybarr.com/media/Using%20Smart%20Charts%20Q&A.pdf
- Mark Graban, "Stop Comparing MLB Stats Year to Year: A Better Way to See Real Trends", Lean Blog, July 2024 (why YoY two-point comparisons mislead): https://www.leanblog.org/2024/07/mlb-stat-comparisons-vs-process-behavior-charts/
- statsmodels `STL` API reference: https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.STL.html and the worked example notebook: https://www.statsmodels.org/stable/examples/notebooks/generated/stl_decomposition.html (unverified; the notebook is shipped in the statsmodels examples)
- Cleveland, R. B., Cleveland, W. S., McRae, J. E. & Terpenning, I. (1990). "STL: A Seasonal-Trend Decomposition Procedure Based on Loess." *Journal of Official Statistics*, 6(1), 3–73.
- Hyndman, R. J. & Athanasopoulos, G. *Forecasting: Principles and Practice*, 3rd ed., OTexts: https://otexts.com/fpp3/ (chapter 3 covers STL and calendar adjustments; direct section link https://otexts.com/fpp3/stl.html unverified)

## 3. New locations with no baseline

### The problem
A location that opened eight weeks ago has eight data points, and all eight come from a process that is *changing by design*: it is ramping. An XmR chart assumes a stable level and uses point-to-point differences as the noise yardstick. During a ramp the point-to-point differences are mostly growth, so (a) the limits are wrong and (b) every rule fires on the trend. Nothing the chart says during the ramp is a statement about the location.

The way out is Wheeler's comparability principle applied along a different axis. Week 5 of the new location is not comparable to its week 4, but it **is** comparable to week 5 of the fifteen locations that opened before it. Indexing time to the opening week turns the openings into a **cohort (vintage) table** with one row per "weeks since opening". The prior cohorts give two things the new location cannot give itself:

1. an **expected ramp shape**: the median, across prior openings, of each site's volume as a fraction of its own mature level at week N;
2. a **noise estimate**: how much prior sites wobbled around that shape week to week, measured as a moving range of their actual÷expected ratios. This noise is *borrowed cross-sectionally*, so limits exist from the new site's very first week.

The chart that results is a ratio chart: $r_N = \dfrac{\text{actual}_N}{\hat{P}\,\cdot\,s(N)}$, where $s(N)$ is the median ramp shape and $\hat{P}$ is the site's expected mature level (from the business plan, or estimated from its first few weeks). If the site ramps like its predecessors, $r_N$ is flat around 1.0 and a standard XmR chart applies to it.

Wheeler's guidance on small samples is the other half of the answer: useful limits can be computed from as few as **5 or 6 values**, treated as provisional; they firm up by 15–20 values, and revising them after about 50 adds nothing (Wheeler, *When Do I Recalculate My Limits?*). Stacey Barr uses 5 as her working minimum. So the borrowed limits bridge weeks 1–5, the site's own provisional limits take over as they become available, and a **graduation rule** decides when the site stops being "new" and gets an ordinary chart.

In [ ]:
# --- fifteen prior openings, each observed for 80 weeks since opening ----------------------------
N_PRIOR, H = 15, 80
age = np.arange(1, H + 1)                                     # weeks since opening (1 = opening week)

def ramp_shape(age, tau):
    """Shared ramp curve: fraction of mature volume reached at a given age (saturating exponential)."""
    return 1 - np.exp(-age / tau)

plateaus = rng.lognormal(np.log(1500), 0.30, N_PRIOR)         # mature weekly volume differs ~2x across sites
taus = rng.normal(8, 1.5, N_PRIOR).clip(4, 14)                # some sites ramp faster than others
prior = pd.DataFrame({f"L{i+1:02d}": np.round(plateaus[i] * ramp_shape(age, taus[i]) * rng.lognormal(0, 0.08, H))
                      for i in range(N_PRIOR)}, index=pd.Index(age, name="weeks since opening"))

# --- one new opening: 20 weeks observed; from week 12 it quietly underperforms by 20% (injected) -------------
OBS_NEW, NEW_TAU, NEW_PLATEAU, UNDER_FROM = 20, 8, 1800, 12
new_true = NEW_PLATEAU * ramp_shape(age[:OBS_NEW], NEW_TAU) * np.where(age[:OBS_NEW] >= UNDER_FROM, 0.80, 1.0)
new = pd.Series(np.round(new_true * rng.lognormal(0, 0.08, OBS_NEW)), index=pd.Index(age[:OBS_NEW], name="weeks since opening"), name="L16 (new)")
print(prior.iloc[:6, :5]); print(); print(new.head(6))

#### (a) Wrong way: raw volume of the new location on an XmR chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_xmr(xmr(new, name="L16 raw volume"), "3.1a All 20 weeks as baseline: the ramp becomes 'signals' at both ends", ax=axes[0], show_mr=False)
plot_xmr(xmr(new, baseline=6, name="L16 raw volume"), "3.1b First 6 weeks as baseline: every later week is 'outside limits'", ax=axes[1], show_mr=False)
for a in axes: a.get_legend().remove(); a.set_xlabel("weeks since opening")
plt.tight_layout()
print("Neither chart can see the 20% underperformance that starts in week", UNDER_FROM, "because the ramp dominates everything.")

#### (b) Right way: the cohort ramp curve and a ratio-to-expected chart with borrowed noise

In [ ]:
# 1. Expected ramp shape from prior openings: each site as a fraction of its own mature level (weeks 27-52)
mature_level = prior.loc[27:52].mean()
shape = prior.div(mature_level, axis=1)                        # columns: sites; rows: age
median_shape = shape.median(axis=1)                            # s(N): expected fraction of mature level at age N

# 2. Borrowed noise: each prior site's ratio to the median shape, then its moving-range average over the first 20 weeks
ratio_prior = shape.div(median_shape, axis=0)                  # actual ÷ expected, per prior site, per age
mr_pooled = ratio_prior.iloc[:OBS_NEW].diff().abs().mean().median()   # median across sites of each site's mR-bar
print(f"borrowed mR-bar of the actual/expected ratio (median over {N_PRIOR} prior sites): {mr_pooled:.3f}")

# 3. Expected mature level for the new site. Two choices: a plan number, or an early-weeks estimate.
PLAN_LEVEL = 1750                                              # pretend the business case said 1,750 / week at maturity
early_est = float((new.iloc[:5] / median_shape.iloc[:5]).median())   # implied mature level from weeks 1-5
print(f"plan mature level: {PLAN_LEVEL}; implied by weeks 1-5: {early_est:.0f}")

fig, ax = plt.subplots(figsize=(11, 4))
for c in shape.columns:
    ax.plot(shape.index, shape[c], color=PAL["muted"], lw=0.8, alpha=0.6)
ax.plot(median_shape.index, median_shape, color=PAL["cl"], lw=2.2, label="median ramp shape s(N)")
ax.plot(new.index, new / early_est, color=PAL["series2"], lw=1.8, marker="o", ms=3.5, label="L16 ÷ its implied mature level")
ax.axvline(UNDER_FROM, color=PAL["muted"], ls=":", lw=0.8)
ax.set_title("3.2 Cohort view: 15 prior openings (grey), their median ramp, and the new site indexed to opening week", loc="left")
ax.set_xlabel("weeks since opening"); ax.set_ylabel("fraction of mature level"); ax.legend(fontsize=8.5)
plt.show()

# 4. Ratio chart for the new site with borrowed limits (central line 1.0 by construction)
expected_new = early_est * median_shape.iloc[:OBS_NEW]
ratio_new = new / expected_new
res_cohort = xmr_fixed(ratio_new, cl=1.0, mr_bar=mr_pooled, name="L16 actual ÷ expected-at-week-N")
plot_xmr(res_cohort, "3.3 Right: actual ÷ expected-at-week-N, limits borrowed from prior cohorts; the week-12 underperformance surfaces",
         units="ratio", annotate={UNDER_FROM: "−20% injected"})
report(res_cohort, injected=UNDER_FROM)

Two things to notice. The injected −20% is caught within a few weeks of starting, with no baseline of the site's own. But the run below 1.0 that begins *before* week 12 is partly an artefact of estimating the mature level from weeks 1–5: a lucky first month sets "expected" too high and every later week looks slightly short. That is the price of the implied-level approach and the reason to prefer a plan figure when one exists.

Using the plan number instead of the early-weeks estimate asks a slightly different question ("is this site tracking its business case?") and is the better choice when the plan is credible, because it does not let a slow first month redefine "expected". Keep both: the ratio-to-plan chart for finance, the ratio-to-implied-level chart for operations.

#### (c) Provisional limits from the site's own first 5–6 points, and how they firm up

In [ ]:
# How wide are the site's OWN limits as points accumulate, compared with the borrowed ones?
# (Only pre-week-12 points are used for the "own" estimate, because from week 12 the site is genuinely different.)
own_rows = []
for k in range(3, UNDER_FROM):
    r = xmr(ratio_new.iloc[:k], name=f"own, {k} pts")
    own_rows.append(dict(points=k, own_mr_bar=r.mr_bar, own_half_width=XMR_X_CONST * r.mr_bar))
own = pd.DataFrame(own_rows)
# what the long-run half-width would be, from a prior site observed for 60 weeks (a stand-in for "truth")
long_run = XMR_X_CONST * ratio_prior.diff().abs().mean().median()

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(own["points"], own["own_half_width"], color=PAL["series"], marker="o", lw=1.6, label="half-width of the site's own limits")
ax.axhline(XMR_X_CONST * mr_pooled, color=PAL["series2"], ls="--", lw=1.4, label="borrowed from prior cohorts (weeks 1-20)")
ax.axhline(long_run, color=PAL["cl"], ls=":", lw=1.2, label="long-run (60-week) cross-site median")
ax.axvspan(5, 6, color=PAL["baseline"], alpha=0.6, label="Wheeler: usable from 5-6 values")
ax.set_title("3.4 Provisional limits: the site's own limits wobble with few points; borrowed limits are a stable stand-in", loc="left")
ax.set_xlabel("number of the site's own points used"); ax.set_ylabel("limit half-width (ratio units)"); ax.legend(fontsize=8.5)
plt.show()
own.round(3).T

Wheeler's rules of thumb, which this plot illustrates: limits from 5–6 values are *usable but provisional* (they can easily be 30–50% off), they become reasonably stable around 15–20 values, and recomputing after 50 is pointless. The deployment consequence: keep the borrowed limits as the operative ones until the site has about 15 of its own ramp-adjusted points without a signal, then switch to its own, and record the switch.

#### (d) A graduation rule: when does a new site get an ordinary XmR chart?
A site graduates from the cohort chart to a standard chart on its raw volume when all three hold:
1. **The ramp is over**: the median shape $s(N)$ is above 0.95 for the ages being charted (the expected week-to-week growth is now smaller than the noise).
2. **No trend remains in its own data**: the least-squares slope of its last 12 ratio values is not distinguishable from zero (t-statistic below 2), and no run signal is active.
3. **Enough points**: at least 12 post-ramp values (15–20 is better) to compute limits that are not purely provisional.

In [ ]:
def graduation_check(site, median_shape, window=12, min_shape=0.95, early_weeks=5):
    """Decide whether a location can leave the cohort chart and get a standard XmR chart.
    site: Series of weekly volume indexed by weeks since opening. Returns (ready, details)."""
    site = site.dropna()
    n = len(site)
    s = median_shape.reindex(site.index)
    P_hat = float((site.iloc[:early_weeks] / s.iloc[:early_weeks]).median())
    ratio = site / (P_hat * s)
    details = {"points": n}
    details["ramp_over"] = bool(n >= window and (s.iloc[-window:] >= min_shape).all())
    y = ratio.iloc[-window:].values if n >= window else ratio.values
    xx = np.arange(len(y))
    if len(y) >= 4:
        slope, intercept = np.polyfit(xx, y, 1)
        resid = y - (slope * xx + intercept)
        se = np.sqrt(resid.var(ddof=2) / ((xx - xx.mean()) ** 2).sum())
        details["trend_t"] = round(float(slope / se), 2) if se > 0 else 0.0
    else:
        details["trend_t"] = None
    r = xmr(ratio.iloc[-window:]) if n >= window else None
    details["run_active"] = bool(r.df["r2"].iloc[-1]) if r is not None else None
    details["no_trend"] = details["trend_t"] is not None and abs(details["trend_t"]) < 2 and not details["run_active"]
    details["enough_points"] = n >= window
    ready = details["ramp_over"] and details["no_trend"] and details["enough_points"]
    return ready, details

ok, det = graduation_check(new, median_shape)
print(f"L16 (new) at week {OBS_NEW}: {'GRADUATE' if ok else 'stay on cohort chart'}  {det}")

# Walk a prior site forward week by week and find the first week it satisfies the rule.
first_ready = None
for weeks in range(12, H + 1):
    ok, det = graduation_check(prior["L03"].iloc[:weeks], median_shape)
    if weeks in (12, 24, 36) or ok and first_ready is None:
        print(f"L03 at week {weeks:2d}: {'GRADUATE' if ok else 'stay on cohort chart'}  {det}")
    if ok and first_ready is None:
        first_ready = weeks
print("L03 first satisfies the graduation rule at week", first_ready, f"(its own ramp constant was {taus[2]:.1f} weeks vs. the cohort median {np.median(taus):.1f})")

# Once graduated, the standard chart starts at the graduation week and uses its first 20 points as the baseline.
# (The 12 weeks that satisfied the no-trend test are evidence for the decision, not part of the new chart.)
start_age = first_ready
post = prior["L03"].loc[start_age:]
res_grad = xmr(post, baseline=20, name="L03 weekly volume (post-ramp)")
plot_xmr(res_grad, f"3.5 After graduation: ordinary XmR on raw volume from week {start_age}, limits from its first 20 post-ramp weeks",
         units="visits / week", show_mr=False)
report(res_grad)

#### (e) Why enterprise charts must split same-store from new-store volume

In [ ]:
# 52 weeks of enterprise volume: 10 mature sites (stable) plus 3 openings at weeks 20, 28 and 36.
WEEKS_E = 52
ew = pd.date_range("2025-01-05", periods=WEEKS_E, freq="W-SUN")
mature_sites = pd.DataFrame({f"M{i+1:02d}": np.round(plateaus[i] * rng.lognormal(0, 0.05, WEEKS_E)) for i in range(10)}, index=ew)
openings = {"N1": 20, "N2": 28, "N3": 36}
new_sites = pd.DataFrame(0.0, index=ew, columns=list(openings))
for name, wk_open in openings.items():
    n_obs = WEEKS_E - wk_open
    new_sites.iloc[wk_open:, new_sites.columns.get_loc(name)] = np.round(1400 * ramp_shape(age[:n_obs], 8) * rng.lognormal(0, 0.08, n_obs))

total = mature_sites.sum(axis=1) + new_sites.sum(axis=1)
same_store = mature_sites.sum(axis=1)                      # "comparable" sites: open for the whole window
fig, axes = plt.subplots(3, 1, figsize=(11, 9))
plot_xmr(xmr(total, baseline=20, name="enterprise total"), "3.6a Enterprise total: a 'growth signal' that is nothing but three openings", ax=axes[0], show_mr=False)
plot_xmr(xmr(same_store, baseline=20, name="same-store total"), "3.6b Same-store total: the established business is flat (as simulated)", ax=axes[1], show_mr=False)
axes[2].stackplot(ew, [new_sites[c] for c in new_sites], labels=list(new_sites), colors=[PAL["series2"], PAL["series3"], PAL["series4"]], alpha=0.85)
axes[2].set_title("3.6c New-store volume, shown separately as the ramps it is (judge each site on its cohort chart)", loc="left"); axes[2].legend(fontsize=8.5, loc="upper left")
for name, wk_open in openings.items():
    for a in axes: a.axvline(ew[wk_open], color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()

### Deployment notes (new locations)
- **Data grain.** Weekly volume keyed by `(location_id, weeks_since_opening)`, built from an `opening_date` table that someone owns. The single most common failure in practice is an opening date that is wrong by two weeks (soft open vs. grand open); define it once and store it.
- **Cohort table.** Materialise the prior-cohort ramp shape and the pooled mR as a small table, versioned by the date it was computed. Recompute it quarterly, or when a cohort of openings graduates, never weekly. Exclude openings that were abnormal for known reasons (a site that opened during a lockdown) and document why.
- **Which cohorts count.** Format, region and size matter: a kiosk does not ramp like a flagship. If you have enough history, build the shape per format; if not, pool and accept wider limits.
- **Governance.** A new site appears on three artefacts: the cohort ratio chart (its own health), the new-store volume panel (enterprise context), and, after graduation, the same-store chart. The graduation decision should be logged with the date and the rule that was satisfied, because it changes the same-store denominator for every later comparison.
- **Retail convention.** "Comparable store" sales in retail usually means stores open at least 12–13 months; the graduation rule here is the statistical version of that convention. Note that it is reached at different ages by different sites (the slow-ramping L03 above needed about 40 weeks): a fixed-age convention is simpler to administer, the rule is more honest.

**Further reading (Section 3)**
- Wheeler, "When Do I Recalculate My Limits?", Quality Digest (1996): https://www.qualitydigest.com/may/spctool.html (limits from few values are temporary; firm up with 15–20; stop revising past ~50)
- Wheeler, "When Should We Compute New Limits?", Quality Digest, April 2012: https://www.qualitydigest.com/node/20205/
- Stacey Barr, "3 Steps to Set a Performance Measure Baseline" (5 values as a working minimum; recalculate only on a signal): https://staceybarr.com/?p=20405
- Stacey Barr, "How to Build an XmR Chart for Your KPI": https://staceybarr.com/?p=6848
- NHS-R `NHSRplotthedots` vignette "Number of points required" (a practical discussion of trial limits from few points): https://cloud.r-project.org/web/packages/NHSRplotthedots/vignettes/number-of-points-required.html
- Wheeler, "Rational Subgrouping", Quality Digest, 2015 (organising data so the chart answers the question; cohorts are a subgrouping choice): https://www.qualitydigest.com/node/26952/

## 4. Sustained growth

### The problem
An established location growing 15–20% a year is not a stable process in the XmR sense: its mean moves every week. A flat central line computed from a baseline is below the data within a few months, so Rule 2 fires continuously, then Rule 1, and the chart's only message is "you are growing", which everyone already knew. Worse, when growth *slows*, the chart stays silent for a long time because the points are still above the old central line.

### Three remedies and what each one asks
1. **Chart the growth rate**: $g_t = \ln(x_t) - \ln(x_{t-1})$ (log period-over-period change). Growth becomes a *level*, so a flat central line is right again. The catch: differencing doubles the noise variance, so this chart is very insensitive to changes in the growth rate when the metric is noisy, and it turns a level shift into a single blip. It is the right chart when the quantity you manage *is* the rate and the series is smooth (compounding SaaS revenue, subscriber bases).
2. **Chart performance relative to an external index**: $x_t / I_t$, where $I_t$ is a market, category or macro index. Growth that the whole market shares cancels; what remains is "us vs. the market". The right chart when the question is attribution and a credible index exists. Its weakness is that it inherits the index's noise and revision history.
3. **Wheeler's trending XmR chart** (half-average method, *Making Sense of Data* ch. 12): split the baseline in two halves, average each half, draw the central line through the two half-averages at their midpoints, and set limits parallel to that line using the moving ranges of the residuals. The chart then asks "are we still on the trend we were on?". The right chart when growth has been steady and you want to see departures from it. Its weakness is exactly that assumption: when growth is not a straight line (S-curves, saturation) the sloped limits produce false signals in the direction of the bend. Because business growth is multiplicative, fit the trend on the **log** of the series and exponentiate the limits.

Data below: 156 weeks. Both we and the market grow at 0.3% a week; at week 100 the market (and we with it) accelerate to 0.8% a week; at week 130 we get an **own +8% level jump** (a pricing or product change) that the market does not share.

In [ ]:
W4 = 156
t4g = np.arange(W4)
wk4g = pd.date_range("2023-01-01", periods=W4, freq="W-SUN")
G1, G2, CHG_WK, JUMP_WK = 0.003, 0.008, 100, 130
log_level = np.cumsum(np.where(t4g < CHG_WK, G1, G2))                               # shared (market) growth path
market = pd.Series(100 * np.exp(log_level + rng.normal(0, 0.015, W4)), index=wk4g, name="market index")
ours_true = 2000 * np.exp(log_level) * np.where(t4g >= JUMP_WK, 1.08, 1.0)
ours = pd.Series(np.round(ours_true * rng.lognormal(0, 0.03, W4)), index=wk4g, name="our weekly orders")
ann4 = {wk4g[CHG_WK]: "growth 0.3%→0.8%/wk (market-wide)", wk4g[JUMP_WK]: "+8% own jump"}

res_flat = xmr(ours, baseline=52, name="our weekly orders")
plot_xmr(res_flat, "4.1 Wrong: flat central line on a growing series. Permanent 'signal', zero information", units="orders / week", annotate=ann4)
report(res_flat)

#### Remedy 1: chart the log growth rate

In [ ]:
g = np.log(ours).diff().dropna() * 100                     # weekly log growth, in %
res_g = xmr(g, baseline=52, name="weekly log growth %")
smooth = pd.Series(np.round(2000 * np.exp(log_level) * np.where(t4g >= JUMP_WK, 1.08, 1.0) * rng.lognormal(0, 0.005, W4)), index=wk4g)
g_smooth = np.log(smooth).diff().dropna() * 100
res_gs = xmr(g_smooth, baseline=52, name="weekly log growth % (0.5% noise series)")
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
plot_xmr(res_g, "4.2a Growth-rate chart on the 3%-noise series: the 0.5pt/wk acceleration is invisible, the +8% jump is one blip", ax=axes[0], show_mr=False, annotate=ann4)
plot_xmr(res_gs, "4.2b Same chart on a smooth series (0.5% noise, e.g. subscription revenue): now it works", ax=axes[1], show_mr=False, annotate=ann4)
plt.tight_layout()
for r in (res_g, res_gs):
    d = r.df; after = d.index[(d.index >= wk4g[CHG_WK]) & d["signal"]]
    print(f"{r.name:45s} sigma={r.sigma:.2f} pts; first signal after the acceleration: {after[0].date() if len(after) else 'none'}")

#### Remedy 2: chart performance relative to a market index

In [ ]:
rel = (ours / market); rel = rel / rel.iloc[:52].mean() * 100            # indexed so the baseline averages 100
res_rel = xmr(rel, baseline=52, name="ours ÷ market index (base=100)")
plot_xmr(res_rel, "4.3 Right (attribution): ours ÷ market. The market-wide acceleration cancels; the own +8% jump is a clean signal",
         units="index", annotate=ann4)
report(res_rel, injected=wk4g[JUMP_WK])

#### Remedy 3: Wheeler's trending XmR (half-average method)

In [ ]:
def xmr_trend(values, index=None, baseline=None, log=True, name="", rules=("r1", "r2", "r3", "r4"), run_length=8):
    """Trending XmR chart by Wheeler's half-average method.

    The baseline points are split into two halves; the central line passes through the two half-averages at
    their midpoints and is extended over all points. Limits are parallel to it, at +/- 2.66 x mR-bar of the
    detrended residuals (Wheeler computes the moving ranges on the raw values; for a steady trend the two differ
    only by the small per-period increment, and the residual version is the safer choice when the slope is steep).
    With log=True the trend is fitted on log(values) so that growth is multiplicative, and the limits are
    exponentiated back to the original scale.
    """
    if index is None and isinstance(values, pd.Series):
        index = values.index
    x = pd.Series(np.asarray(values, dtype=float), index=(index if index is not None else pd.RangeIndex(len(values))))
    n = len(x); pos = np.arange(n)
    bmask = _baseline_mask(baseline, n)
    y = np.log(x.values) if log else x.values
    bp = pos[bmask]; by = y[bmask]
    h = len(bp) // 2
    a1, a2 = by[:h].mean(), by[h:].mean()
    m1, m2 = bp[:h].mean(), bp[h:].mean()
    slope = (a2 - a1) / (m2 - m1)
    cl_y = a1 + slope * (pos - m1)
    resid = y - cl_y
    mr_resid = pd.Series(resid).diff().abs().values
    mr_in_base = bmask & np.roll(bmask, 1); mr_in_base[0] = False
    mr_bar = float(np.nanmean(mr_resid[mr_in_base]))
    sigma = XMR_X_CONST * mr_bar / 3.0
    flags = detect_signals(resid, sigma, rules, run_length)
    back = np.exp if log else (lambda v: v)
    df = pd.DataFrame({"x": x.values, "mr": mr_resid, "cl": back(cl_y), "unpl": back(cl_y + XMR_X_CONST * mr_bar),
                       "lnpl": back(cl_y - XMR_X_CONST * mr_bar), "sigma": sigma, "mr_cl": mr_bar,
                       "mr_url": XMR_MR_CONST * mr_bar, "baseline": bmask}, index=x.index)
    for k, v in flags.items():
        df[k] = v
    df["mr_r1"] = (df["mr"] > df["mr_url"])
    df["signal"] = df[["r1", "r2", "r3", "r4"]].any(axis=1)
    obj = XmR(df, float(df["cl"].iloc[-1]), mr_bar, sigma, name)
    obj.unpl, obj.lnpl = float(df["unpl"].iloc[-1]), float(df["lnpl"].iloc[-1])   # at the last point
    obj.slope_per_period = slope
    return obj


res_tr = xmr_trend(ours, baseline=52, log=True, name="our weekly orders (trending)")
print(f"fitted trend: {100 * res_tr.slope_per_period:.2f}% per week (true baseline growth {100 * G1:.2f}%)")
plot_xmr(res_tr, "4.4 Right (steady growth): trending XmR, log-linear central line from the first 52 weeks. Departure from trend after week 100",
         units="orders / week", annotate=ann4)
report(res_tr, injected=wk4g[CHG_WK])

Note what each chart *said*. The trending chart correctly reported "we have left the trend we were on" after week 100, but it cannot say whether that is us or the market; the index-relative chart answered that (it was the market) and isolated the week-130 jump as ours. Different question, different chart. Note also that the mR panel of the trending chart is computed on the detrended residuals, which is why it stays flat while the raw series climbs.

#### The weakness of the trending chart when growth is not steady

In [ ]:
# An S-curve: fast early growth that saturates. Trend fitted on the steep part predicts growth that never comes.
tt = np.arange(120)
s_curve = 3000 / (1 + np.exp(-(tt - 40) / 12))
s_series = pd.Series(np.round(s_curve * rng.lognormal(0, 0.03, len(tt))), index=pd.date_range("2023-01-01", periods=len(tt), freq="W-SUN"))
res_s = xmr_trend(s_series, baseline=40, log=True, name="S-curve volume")
plot_xmr(res_s, "4.5 Trending XmR on an S-curve: limits fitted during the steep phase fire false 'decline' signals as growth saturates",
         units="orders / week", show_mr=False)
print("flagged points after week 60 (all false, the process simply matured):", int(res_s.df['signal'].iloc[60:].sum()), "of 60")

When growth is not steady, the honest options are: shorten the baseline and re-fit the trend at documented phase changes (Section 5), switch to the growth-rate or index-relative chart, or model the curve explicitly (which is forecasting, not process monitoring, and should be labelled as such).

#### Decomposition: component charts localise a change without knowing the cause
Total orders = new-customer orders + returning-customer orders. Both grow with the market; at week 110 **new-customer orders drop 15%** (say a paid-search change) while returning customers are untouched. New customers are about a fifth of the total, so the total moves by roughly 3%, about one sigma of its own noise. Compare the printed evidence: the total may produce a zone-rule or run flag at some point (a one-sigma shift eventually does), but it never produces the unambiguous Rule 1 evidence that the component chart produces week after week, and it cannot say where to look. Each series is charted as its ratio to the market index (Remedy 2), which removes the shared growth, so a plain XmR applies; fitting a trend to a series that no longer has one only manufactures false runs. (The occasional zone-rule flag on the untouched component is the usual false-alarm cost of Rules 3 and 4.)

In [ ]:
new_cust = pd.Series(np.round(500 * np.exp(log_level) * np.where(t4g >= 110, 0.85, 1.0) * rng.lognormal(0, 0.05, W4)), index=wk4g)
ret_cust = pd.Series(np.round(1900 * np.exp(log_level) * rng.lognormal(0, 0.025, W4)), index=wk4g)
total_c = new_cust + ret_cust
parts = {"total orders": total_c, "new-customer orders": new_cust, "returning-customer orders": ret_cust}
fig, axes = plt.subplots(3, 1, figsize=(11, 9.5))
for ax, (k, s) in zip(axes, parts.items()):
    r = xmr(s / market * 100, baseline=52, name=k + " ÷ market")   # ratio to market has no trend left, so a plain XmR is the right chart
    plot_xmr(r, f"4.6 {k} ÷ market index", ax=ax, show_mr=False)
    ax.axvline(wk4g[110], color=PAL["muted"], ls=":", lw=0.8)
    after = r.df.index[(r.df.index >= wk4g[110]) & r.df["signal"]]
    delay = (after[0] - wk4g[110]).days // 7 if len(after) else None
    print(f"{k:28s} first signal after week 110: {after[0].date() if len(after) else 'none'}  (delay {delay} weeks); Rule 1 points after: {int(r.df['r1'][r.df.index >= wk4g[110]].sum())}")
plt.tight_layout()

### When to use which (growth)
| Situation | Chart | Why |
|---|---|---|
| Steady growth, question is "are we still on track?" | Trending XmR (log) | Limits follow the trend; departures show as signals |
| Growth is itself the managed quantity, series is smooth | Log growth-rate chart | Growth becomes a level; a flat chart is valid |
| Question is attribution ("us or the market?"), credible index exists | Index-relative chart | Shared growth cancels |
| Growth bends (launch S-curves, saturation) | Phase the trend chart, or growth-rate chart | A straight trend will produce false signals at the bend |
| Need to know *where* a change came from | Component charts (same method as the total) | A 20% change in a 30% component is a 6% change in the total |

### Deployment notes (growth)
- **Re-fit the trend only at documented phase changes**, never on a rolling window; a rolling trend is a moving average in disguise (Section 0).
- **Store the trend parameters** (baseline window, slope per period, log or linear) in the registry; the chart's subtitle should print them.
- **Index choice** is a governance decision: who publishes it, how often it is revised, and what you do when it is revised (re-run history, annotate).
- **Components must be additive** (new + returning = total) and defined by the same system of record as the total, or the decomposition will not reconcile and the review will argue about the data instead of the signal.

**Further reading (Section 4)**
- Wheeler, D. J. *Making Sense of Data*, SPC Press, 2003, ch. 12 (the half-average trending chart and its limits).
- Stacey Barr, "Interpreting Signals in Trending KPIs" (a step-by-step trending XmR chart; she also warns that regression trendlines are poor at detecting signals): https://www.staceybarr.com/?p=19204
- Stacey Barr, "3 Essential Signals to Look for in Your KPIs": https://staceybarr.com/?p=2746
- Mark Graban, *Measures of Success: React Less, Lead Better, Improve More*, Constancy, 2018 (a manager's treatment of PBCs, including trending data); chapter summary: https://www.leanblog.org/2024/02/measures-of-success-book-summary/
- Cedric Chin, "Process Behaviour Charts: More Than You Need To Know", Commoncog: https://commoncog.com/process-behaviour-charts-more-than-you-need/ and "Becoming Data Driven, From First Principles": https://commoncog.com/becoming-data-driven-first-principles/ (the business-review framing used by Amazon's WBR)

## 5. Limit recalculation and phase charts

### The problem
Limits are a *prediction*: "if nothing changes, next week's value will land in here". Two opposite mistakes destroy that prediction.

1. **Recomputing too often.** Software that recalculates the limits every time a point is added is "unnecessary at best and may mislead on occasion" (Wheeler, *When Should We Compute New Limits?*). A chart that re-fits itself every week can never say that something changed, because it has already absorbed the change. Stacey Barr's rule is blunter: adding new data is *never* by itself a reason to recompute; a signal is.
2. **Never recomputing, or recomputing over a shock.** A pandemic quarter, a billing outage, or a one-off promotion that is left inside the limit calculation inflates the average moving range. The limits then widen permanently, and the chart goes blind to every ordinary-sized change that follows. The damage is quantified below.

### When to compute new limits
Wheeler's four questions (paraphrased from his 1996 column and the 2012 follow-up): is there a signal that the process has changed? is the reason for the change known? is the new behaviour expected to continue? is the new level the one you want to maintain? If all four are yes, start a new **phase** from the point where the change began and compute its limits from the data since then, treating the first 5–6 values as provisional and firming up by 15–20 (Section 3). If the cause is known but temporary (a shock), do *not* start a phase: annotate the points, exclude them from any limit calculation, and keep the old limits running so that the recovery can be judged against them.

The resulting **phase chart** has limits that step at documented dates. The documentation is the point: every step should be traceable to a reason, a decision, and a decider.

Data below: 160 weeks of weekly revenue. Pricing change at week 60 (+15%, intentional). External shock from week 100 to 108 (−50%, recovering by week 116). A **+7% shift of unknown cause at week 140** that the chart is supposed to catch.

In [ ]:
W5 = 160
t5 = np.arange(W5); wk5 = pd.date_range("2023-01-01", periods=W5, freq="W-SUN")
PRICE_WK, SHOCK_START, SHOCK_END, UNKNOWN_WK = 60, 100, 116, 140
level5 = 10_000 * np.where(t5 >= PRICE_WK, 1.15, 1.0) * np.where(t5 >= UNKNOWN_WK, 1.07, 1.0)
shock = np.ones(W5)
shock[SHOCK_START:109] = 0.5 * rng.lognormal(0, 0.20, 9)              # nine weeks at about half volume, and volatile
shock[109:SHOCK_END + 1] = np.linspace(0.55, 1.0, SHOCK_END + 1 - 109) * rng.lognormal(0, 0.08, SHOCK_END + 1 - 109)   # noisy recovery
rev = pd.Series(np.round(level5 * shock * rng.lognormal(0, 0.03, W5)), index=wk5, name="weekly revenue")
ann5 = {wk5[PRICE_WK]: "pricing +15% (intentional)", wk5[SHOCK_START]: "external shock", wk5[UNKNOWN_WK]: "+7%, cause unknown"}

# WRONG 1: one set of limits from everything up to today (no phase at the price change)
res_all = xmr(rev, name="revenue, limits from all 160 weeks")
# WRONG 2: limits re-fitted on a trailing 26-week window every week (what many BI tools do)
roll_cl = rev.rolling(26).mean(); roll_mr = rev.diff().abs().rolling(26).mean()
# WRONG 3: phased correctly at the price change, but the shock weeks were left INSIDE the phase-2 limit calculation
res_shock_in = xmr(rev.iloc[PRICE_WK:], name="phase 2, limits over weeks 60-159 incl. shock")

fig, axes = plt.subplots(3, 1, figsize=(11, 10.5))
plot_xmr(res_all, "5.1a Wrong: no phase at the price change. The blended centre line means nothing and the chart reports the price rise forever",
         ax=axes[0], show_mr=False, annotate=ann5)
axes[1].plot(wk5, rev, color=PAL["series"], lw=1.3, marker="o", ms=2.8, label="revenue")
axes[1].plot(wk5, roll_cl, color=PAL["cl"], lw=1.2, label="trailing 26-wk central line")
axes[1].plot(wk5, roll_cl + XMR_X_CONST * roll_mr, color=PAL["limit"], ls="--", lw=1, label="trailing limits")
axes[1].plot(wk5, roll_cl - XMR_X_CONST * roll_mr, color=PAL["limit"], ls="--", lw=1)
axes[1].set_title("5.1b Wrong: limits re-fitted every week on a trailing window. They chase the data and can never say 'changed'", loc="left")
axes[1].legend(fontsize=8.5, loc="upper left", bbox_to_anchor=(1.01, 1.0))
plot_xmr(res_shock_in, "5.1c Wrong: phased at the price change, but the shock left inside the limits: wide limits, depressed centre line",
         ax=axes[2], show_mr=False, annotate=ann5)
plt.tight_layout()

clean = rev.iloc[PRICE_WK:SHOCK_START]
clean_mr, clean_cl = float(clean.diff().abs().mean()), float(clean.mean())
print(f"phase-2 mR-bar from clean weeks 60-99: {clean_mr:,.0f}   with the shock inside (weeks 60-159): {res_shock_in.mr_bar:,.0f}  "
      f"-> limits {res_shock_in.mr_bar / clean_mr:.2f}x wider")
print(f"phase-2 centre line clean: {clean_cl:,.0f}   with the shock inside: {res_shock_in.cl:,.0f}  (depressed by {100 * (1 - res_shock_in.cl / clean_cl):.1f}%)")
d = res_shock_in.df
recovered = d.iloc[SHOCK_END + 1 - PRICE_WK: UNKNOWN_WK - PRICE_WK]
after = d.iloc[UNKNOWN_WK - PRICE_WK:]
print(f"chart 5.1c flags {int(recovered['signal'].sum())} of {len(recovered)} recovered-but-unchanged weeks (117-139) and "
      f"{int(after['signal'].sum())} of {len(after)} weeks after the +7% shift: it cannot tell the shift from the recovery")

#### Right way: a documented phase chart

In [ ]:
def xmr_phases(values, index=None, phase_starts=(0,), baseline_len=20, exclude=None, rules=("r1", "r2", "r3", "r4"), name=""):
    """XmR chart whose limits are recomputed at each phase start.

    phase_starts : positions (0-based) where a new phase begins; the first should be 0
    baseline_len : limits of each phase come from its first `baseline_len` non-excluded points (None = whole phase)
    exclude      : boolean mask of points to keep OUT of every limit calculation (known shocks); they are still plotted
    """
    if index is None and isinstance(values, pd.Series):
        index = values.index
    x = pd.Series(np.asarray(values, dtype=float), index=(index if index is not None else pd.RangeIndex(len(values))))
    n = len(x)
    excl = np.zeros(n, dtype=bool) if exclude is None else np.asarray(exclude, dtype=bool)
    starts = list(phase_starts) + [n]
    parts = []
    for a, b in zip(starts[:-1], starts[1:]):
        seg = x.iloc[a:b]
        ok = ~excl[a:b]
        bmask = ok.copy()
        if baseline_len is not None:                      # first `baseline_len` usable points of the phase
            keep = np.cumsum(ok) <= baseline_len
            bmask = ok & keep
        r = xmr(seg, baseline=bmask, rules=rules)
        d = r.df.copy(); d["phase"] = len(parts) + 1
        parts.append(d)
    df = pd.concat(parts)
    df["excluded"] = excl
    df.loc[df["excluded"], ["r1", "r2", "r3", "r4", "signal"]] = False      # known-cause points are not "signals"
    obj = XmR(df, float(df["cl"].iloc[-1]), float(df["mr_cl"].iloc[-1]), float(df["sigma"].iloc[-1]), name)
    obj.unpl, obj.lnpl = float(df["unpl"].iloc[-1]), float(df["lnpl"].iloc[-1])
    return obj


# The decision log is part of the chart. Each row is a governance decision, not a statistical one.
phase_log = pd.DataFrame([
    dict(date=wk5[0].date(),          event="chart created",            action="phase 1 limits from first 20 weeks",            decided_by="FP&A lead"),
    dict(date=wk5[PRICE_WK].date(),   event="price increase +15%",      action="phase 2 from this week (intentional, permanent)", decided_by="pricing committee"),
    dict(date=wk5[SHOCK_START].date(),event="external shock",           action="exclude weeks 100-116 from limits; keep phase 2 limits; annotate", decided_by="weekly review"),
    dict(date=wk5[SHOCK_END + 1].date(), event="recovery complete",     action="phase 3 from week 117, provisional limits from its first 20 weeks", decided_by="weekly review"),
])
display(phase_log)

excluded = (t5 >= SHOCK_START) & (t5 <= SHOCK_END)
res_ph = xmr_phases(rev, phase_starts=(0, PRICE_WK, SHOCK_END + 1), baseline_len=20, exclude=excluded, name="weekly revenue (phases)")
plot_xmr(res_ph, "5.2 Right: phase chart. Limits step at documented dates; shock weeks are excluded from limits; week-140 shift is caught",
         units="revenue / week", annotate=ann5, step_limits=True)
report(res_ph, injected=wk5[UNKNOWN_WK])
first = res_ph.df.index[(t5 >= UNKNOWN_WK) & res_ph.df["signal"].values]
print("phase 3 mR-bar:", round(res_ph.mr_bar), "vs shock-inside mR-bar:", round(res_shock_in.mr_bar),
      "| weeks from the +7% shift to the first signal:", (first[0] - wk5[UNKNOWN_WK]).days // 7 if len(first) else "none",
      "| false flags in the recovered weeks 117-139:", int(res_ph.df["signal"].values[SHOCK_END + 1:UNKNOWN_WK].sum()))

### Deployment notes (recalculation)
- **Never automatic.** Limits change only by a logged decision. The log (date, event, action, decider) lives next to the metric definition and is printed on the chart or linked from it.
- **Trigger, not schedule.** A signal plus a known, persistent, intended cause starts a phase. "It has been a year" does not. "Finance re-forecast" does not.
- **Shocks are excluded, not phased.** Keep the old limits running through a temporary shock so that recovery is visible; exclude the shock weeks from any later baseline; annotate them. If the shock turns out to be permanent (the market shrank for good), it becomes a phase, with its start set at the shock's end.
- **Provisional first.** New-phase limits from the first 5–6 points are usable, from 15–20 stable; mark the phase "provisional" in the chart subtitle until then and avoid firm conclusions from Rules 3 and 4 in that window. (In the example above the phase-3 limits came out wider than phase 2's purely from 20-point sampling; that is normal and is exactly why they are provisional.)
- **Keep history.** Old phases are not deleted; a chart that only shows the current phase loses the context that makes the step credible.

**Further reading (Section 5)**
- Wheeler, "When Do I Recalculate My Limits?", Quality Digest, 1996: https://www.qualitydigest.com/may/spctool.html
- Wheeler, "When Should We Compute New Limits?", Quality Digest, April 2012: https://www.qualitydigest.com/node/20205/
- Stacey Barr, "CASE STUDY: 3 Common Mistakes To Avoid With Your XmR Charts (Part 3)" (recalculating at the start of a run signal): https://staceybarr.com/?p=2795
- Stacey Barr, "3 Steps to Set a Performance Measure Baseline": https://staceybarr.com/?p=20405
- Wheeler, "Good Limits From Bad Data", Quality Digest, March 1997 (why the moving range, and the median-moving-range alternative with constant 3.145 when extreme ranges contaminate the average): https://www.qualitydigest.com/march97/html/spctool.html

## 6. Pain points common to other businesses

One compact example each. The pattern is always the same: identify why successive values stopped being comparable, restore comparability, chart that.

### 6.1 Rates with variable denominators (conversion rate)
A weekly conversion rate is conversions ÷ visitors. When visitor counts vary a lot week to week, the sampling noise of the rate varies with them: a 5% rate on 200 visitors wobbles by ±1.5 points from sampling alone, on 2,000 visitors by ±0.5. Three charts compete:
- **A classical p-chart** sets limits from the binomial formula $\bar p \pm 3\sqrt{\bar p(1-\bar p)/n_i}$. It assumes the *only* variation is sampling. Real funnels also vary between weeks for a hundred other reasons (**overdispersion**), so on large denominators the p-chart limits hug the mean and fire constantly (Laney 2002; Wheeler, *What About p-Charts?*).
- **An XmR chart on the rate** uses the empirical moving range, so it captures the real between-week variation (Wheeler's recommendation for most people). Its weakness: it gives the same limits to a 200-visitor week and a 2,000-visitor week.
- **Laney's p′ chart** fixes both: standardise each week to a z-score with the binomial sigma, measure the *actual* dispersion of the z-scores with a moving range ($\sigma_z$), and scale the binomial limits by it: $\bar p \pm 3\,\sigma_z\sqrt{\bar p(1-\bar p)/n_i}$. If the data really are binomial, $\sigma_z \approx 1$ and it reduces to the p-chart; if all $n_i$ are equal it reduces to the XmR chart.

**Minimum-denominator rule.** Whatever chart you use, a week with too few trials carries almost no information and its wild value will either cause a false alarm (XmR) or be excused by enormous limits (p′). Two checks: (1) a **hard floor**: pool any period whose denominator gives fewer than about 5 expected events ($n_i \bar p < 5$) with the next period until it clears; this is the same "chunky data" problem Wheeler describes for counts. (2) A **grain check**: compare the binomial sampling sigma at the typical denominator with the chart's sigma; if sampling noise is most of the chart's noise, the rate is being charted at too fine a grain, and a 2-week or monthly grain will see real changes sooner than a noisy weekly one.

In [ ]:
def laney_p_prime(successes, trials, index=None, baseline=None, name="Laney p′"):
    """Laney p' chart. Returns an XmR-like object with per-point limits (they vary with n_i)."""
    s = np.asarray(successes, dtype=float); n = np.asarray(trials, dtype=float)
    idx = index if index is not None else pd.RangeIndex(len(s))
    bmask = _baseline_mask(baseline, len(s))
    p = s / n
    pbar = s[bmask].sum() / n[bmask].sum()
    sig_p = np.sqrt(pbar * (1 - pbar) / n)                     # binomial sigma for each week
    z = (p - pbar) / sig_p
    mr_z = pd.Series(z).diff().abs().values
    sigma_z = np.nanmean(mr_z[1:][bmask[1:]]) / 1.128          # the Laney adjustment: actual dispersion of z
    half = 3 * sigma_z * sig_p
    df = pd.DataFrame({"x": p, "mr": pd.Series(p).diff().abs().values, "cl": pbar, "unpl": pbar + half, "lnpl": pbar - half,
                       "sigma": half / 3, "mr_cl": np.nan, "mr_url": np.nan, "baseline": bmask}, index=idx)
    flags = detect_signals(df["x"].values - pbar, float(np.median(df["sigma"][bmask])), ("r2",))   # run rule at the median sigma
    df["r1"] = (df["x"] > df["unpl"]) | (df["x"] < df["lnpl"])
    df["r2"], df["r3"], df["r4"] = flags["r2"], False, False
    df["mr_r1"] = False; df["signal"] = df[["r1", "r2"]].any(axis=1)
    obj = XmR(df, pbar, np.nan, float(np.median(df["sigma"])), name); obj.sigma_z = sigma_z
    obj.unpl, obj.lnpl = float(df["unpl"].iloc[-1]), float(df["lnpl"].iloc[-1])
    return obj


# 60 weeks: visitors vary 10x; true rate 5% with real between-week variation (overdispersion); +1.2 pt shift at week 40
nwk = 60
wk6 = pd.date_range("2025-01-05", periods=nwk, freq="W-SUN")
visitors = rng.integers(60, 2500, nwk)
p_true = np.clip(rng.normal(0.050, 0.004, nwk), 0.01, 0.2) + np.where(np.arange(nwk) >= 40, 0.012, 0)
conversions = rng.binomial(visitors, p_true)
rate = pd.Series(conversions / visitors, index=wk6, name="conversion rate")

# classical p-chart (sigma_z forced to 1)
pbar0 = conversions[:40].sum() / visitors[:40].sum()
p_half = 3 * np.sqrt(pbar0 * (1 - pbar0) / visitors)
res_x = xmr(rate, baseline=40, name="XmR on the rate")
res_lp = laney_p_prime(conversions, visitors, index=wk6, baseline=40)

fig, axes = plt.subplots(3, 1, figsize=(11, 9.5), sharex=True)
axes[0].plot(wk6, rate, color=PAL["series"], marker="o", ms=3, lw=1.2); axes[0].axhline(pbar0, color=PAL["cl"])
axes[0].plot(wk6, pbar0 + p_half, color=PAL["limit"], ls="--", lw=1); axes[0].plot(wk6, pbar0 - p_half, color=PAL["limit"], ls="--", lw=1)
out = (rate > pbar0 + p_half) | (rate < pbar0 - p_half)
axes[0].plot(wk6[out], rate[out], "X", color=PAL["r1"], ms=8)
axes[0].set_title(f"6.1a Classical p-chart: binomial limits hug the mean on big weeks; {int(out[:40].sum())} false alarms in the baseline", loc="left")
plot_xmr(res_x, "6.1b XmR on the rate: honest about between-week variation, but one width for every denominator", ax=axes[1], show_mr=False)
plot_xmr(res_lp, f"6.1c Laney p′: limits widen for small weeks and scale by σz = {res_lp.sigma_z:.2f}", ax=axes[2], show_mr=False)
for a in axes: a.axvline(wk6[40], color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()
n_floor = 5 / pbar0                                                    # hard floor: at least 5 expected conversions
samp_sigma = np.sqrt(pbar0 * (1 - pbar0) / np.median(visitors))         # sampling noise at the typical denominator
print(f"hard floor: n_i >= {n_floor:,.0f} visitors; weeks below it: {int((visitors < n_floor).sum())} (pool each with its neighbour)")
print(f"grain check: sampling sigma at the median denominator = {100 * samp_sigma:.2f} pts vs chart sigma = {100 * res_x.sigma:.2f} pts "
      f"-> sampling is {100 * samp_sigma ** 2 / res_x.sigma ** 2:.0f}% of the chart's variance; "
      f"{'consider a coarser grain' if samp_sigma ** 2 / res_x.sigma ** 2 > 0.5 else 'weekly grain is fine'}")

*Deployment:* store numerator and denominator, never just the rate; decide the chart per metric (XmR by default, p′ when denominators vary more than about 3×); apply the pooling rule in SQL before charting.
*Reading:* Laney, D. B. (2002), "Improved Control Charts for Attributes", *Quality Engineering* 14(4), 531–537, https://doi.org/10.1081/QEN-120003555; Laney, "P-Charts and U-Charts Work (But Only Sometimes)", Quality Digest, Feb 2007: https://www.qualitydigest.com/feb07/departments/what_works.shtml; Wheeler, "What About p-Charts?", Quality Digest, Sept 2011: https://www.qualitydigest.com/node/18604/; Minitab, "Overview for Laney P′ Chart": https://support.minitab.com/minitab/help-and-how-to/quality-and-process-improvement/control-charts/how-to/attributes-charts/laney-p-chart/before-you-start/overview and "Overdispersion and underdispersion": https://support.minitab.com/minitab/help-and-how-to/quality-and-process-improvement/control-charts/supporting-topics/understanding-attributes-control-charts/overdispersion-and-underdispersion

### 6.2 Rare events (churn, outages): chart time between events, not counts
With an average of 0.3 churned enterprise accounts per week, most weeks are zero, and a count chart has a lower limit below zero and an upper limit around 2: a week with 2 churns is "normal" and a week with 3 is a crisis, which is useless. Wheeler's rule: when the average count per period falls below about 1, counts stop carrying information. Chart the **time between events** instead (days from one churn to the next); more churn shows up as *shorter* gaps and a run below the central line. Gaps are right-skewed, so the lower limit can go negative; treat 0 as the floor, and if you want symmetric limits use the t-chart transform $y = t^{1/3.6}$ (Provost & Murray) or chart the rate $1/t$ (Wheeler).

In [ ]:
# 3 years of daily data; churn events are Poisson at 0.3/week, doubling after day 730 (injected)
ndays = 3 * 365
day_rate = np.where(np.arange(ndays) < 730, 0.3 / 7, 0.6 / 7)
events_daily = rng.poisson(day_rate)
dates6 = pd.date_range("2023-01-01", periods=ndays, freq="D")
event_days = dates6[np.repeat(np.arange(ndays), events_daily)]
weekly_counts = pd.Series(events_daily, index=dates6).resample("W-SUN").sum()
gaps = pd.Series(np.diff(event_days.values).astype("timedelta64[D]").astype(int), index=event_days[1:], name="days between churn events")

fig, axes = plt.subplots(2, 1, figsize=(11, 7))
plot_xmr(xmr(weekly_counts, baseline=104, name="weekly churn count"), "6.2a Wrong: weekly counts of a rare event. Average < 1, lower limit < 0, every value is 'normal'", ax=axes[0], show_mr=False)
res_gap = xmr(gaps, baseline=int((event_days[1:] < pd.Timestamp("2025-01-01")).sum()), name="days between events")
plot_xmr(res_gap, "6.2b Right: days between events. Doubling the churn rate halves the gaps; a run below the central line", ax=axes[1], show_mr=False, ylim=(0, None))
for a in axes: a.axvline(dates6[730], color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()
print("events:", len(event_days), "| mean gap before/after day 730:",
      round(float(gaps[gaps.index < dates6[730]].mean()), 1), "/", round(float(gaps[gaps.index >= dates6[730]].mean()), 1), "days")

*Deployment:* the x-axis is the event sequence, not the calendar, so the chart updates only when an event happens; show "days since last event" as a live indicator between points. Censoring matters: the open interval since the last event is not a data point, but if it exceeds the upper limit that is itself good news worth stating.
*Reading:* Wheeler, "Charts for Rare Events", Quality Digest, Dec 1996: https://www.qualitydigest.com/dec96/toolkit.html; Wheeler, "Working with Rare Events", Quality Digest, Oct 2011: https://www.qualitydigest.com/node/18838/; Provost, L. P. & Murray, S. K., *The Health Care Data Guide*, 2nd ed., Jossey-Bass, 2022 (T chart and G chart chapters); Benneyan, J. C. (2001), "Number-between g-type statistical quality control charts for monitoring adverse events", *Health Care Management Science* 4, 305–318; BPI Consulting, "Rare Events and X-mR Charts": https://www.spcforexcel.com/knowledge/variable-control-charts/rare-events-and-x-mr-charts/

### 6.3 Bounded metrics near a ceiling (CSAT, uptime)
A satisfaction score averaging 96% cannot exceed 100%, and its noise is asymmetric: it can fall 10 points but rise only 4. A symmetric XmR chart on the raw percentage puts the upper limit above 100% (meaningless) and the lower limit too close (false alarms on the downside) or too far (missed drops), depending on the sample. Two fixes: chart the **complement** (dissatisfied %, downtime minutes), which turns the ceiling into a floor at zero and often into a small-count metric that Section 6.2's logic applies to; or chart the **logit**, $\ln\frac{p}{1-p}$, which maps (0,1) to the whole real line and makes the noise roughly symmetric, then back-transform the limits for display. Wheeler is sceptical of transformations in general ("Avoiding Statistical Jabberwocky"); the logit is defensible here because the bound is a known property of the measurement, not a convenience.

In [ ]:
nw = 60; wk63 = pd.date_range("2025-01-05", periods=nw, freq="W-SUN")
n_resp = 200
p_sat = np.where(np.arange(nw) >= 40, 0.93, 0.96) + rng.normal(0, 0.006, nw)        # real −3 pt drop at week 40
csat = pd.Series(rng.binomial(n_resp, p_sat) / n_resp * 100, index=wk63, name="CSAT %")

def logit(p): return np.log(p / (1 - p))
def inv_logit(z): return 1 / (1 + np.exp(-z))

res_raw6 = xmr(csat, baseline=40, name="CSAT % (raw)")
res_lg = xmr(logit(csat / 100), baseline=40, name="logit(CSAT)")
# back-transform the logit chart to % for display
d = res_lg.df.copy()
for c in ("x", "cl", "unpl", "lnpl"):
    d[c] = inv_logit(d[c]) * 100
d["sigma"] = np.nan
res_lg_disp = XmR(d, inv_logit(res_lg.cl) * 100, np.nan, np.nan, "CSAT % (limits via logit)")
res_comp = xmr(100 - csat, baseline=40, name="dissatisfied %")

fig, axes = plt.subplots(3, 1, figsize=(11, 9.5), sharex=True)
plot_xmr(res_raw6, f"6.3a Wrong: raw CSAT. Upper limit {res_raw6.unpl:.1f}% {'is above the 100% ceiling' if res_raw6.unpl > 100 else 'crowds the 100% ceiling'}; symmetric limits on an asymmetric metric", ax=axes[0], show_mr=False)
axes[0].axhline(100, color=PAL["r1"], lw=0.8, ls=":")
plot_xmr(res_lg_disp, "6.3b Option 1: limits on the logit scale, back-transformed: inside the bounds and asymmetric, but note the generous lower limit", ax=axes[1], show_mr=False)
plot_xmr(res_comp, "6.3c Option 2 (recommended): chart the complement (dissatisfied %), a floor at zero instead of a ceiling", ax=axes[2], show_mr=False)
for a in axes: a.axvline(wk63[40], color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()

Look closely at 6.3b: the logit stretches the top of the scale, so the weeks nearest the ceiling dominate the moving ranges and the back-transformed *lower* limit comes out generous. The logit fixes the cosmetic problem (a limit above 100%) at the cost of downside sensitivity. The complement (6.3c) has neither problem and is the better default; all three charts catch the −3 point drop through the run rule.

*Deployment:* register the transformation with the metric (Section 7) and always display in business units. For uptime, chart downtime minutes per period (complement) rather than 99.9x%: the numbers reviewers read should be the ones that move.
*Reading:* Wheeler, "Avoiding Statistical Jabberwocky", Quality Digest, Oct 2009 (the case against routine transformation): https://www.qualitydigest.com/node/9126/; Wheeler, "Hazardous Ideas About Charts for Individual Values", Quality Digest, 2026 (normality is not required for the limits to work): https://www.qualitydigest.com/inside/statistics-article/hazardous-ideas-about-charts-individual-values-072726.html

### 6.4 Long-lag outcomes (retention): chart cohorts at a fixed age
90-day retention for the signup cohort of week *w* is not known until 13 weeks after *w*. The common mistake is to chart "retention so far" for every cohort including the immature ones: the last 13 cohorts are observed at younger ages, where retention is naturally higher (or lower, depending on the metric), so the chart ends with a fake trend every single week. Restore comparability by indexing on **cohort age**: chart each cohort's retention at exactly 90 days and accept the 13-week lag. If you need an earlier warning, add a second chart at a fixed *early* age (day 14) as a leading proxy, and validate that it predicts the 90-day figure.

In [ ]:
n_coh = 80; TODAY = n_coh - 1                                   # cohorts 0..79 (weekly signups); we are at the end of week 79
coh_idx = pd.date_range("2024-06-02", periods=n_coh, freq="W-SUN")
r90_true = np.where(np.arange(n_coh) >= 50, 0.34, 0.40)         # real drop for cohorts from week 50 (injected)
def survival(age_days, r90):                                   # retention curve through (90, r90)
    return r90 ** ((age_days / 90) ** 0.6)
retention_so_far, retention_90, retention_14 = [], [], []
for c in range(n_coh):
    age = min(90, (TODAY - c) * 7)                               # how old is cohort c today
    noise = rng.normal(0, 0.012)
    retention_so_far.append(survival(age, r90_true[c]) + noise)
    retention_90.append(survival(90, r90_true[c]) + noise if age >= 90 else np.nan)
    retention_14.append(survival(14, r90_true[c]) + rng.normal(0, 0.012))
sofar = pd.Series(retention_so_far, index=coh_idx, name="retention so far") * 100
r90 = pd.Series(retention_90, index=coh_idx, name="90-day retention").dropna() * 100
r14 = pd.Series(retention_14, index=coh_idx, name="14-day retention").dropna() * 100

fig, axes = plt.subplots(3, 1, figsize=(11, 9.5))
plot_xmr(xmr(sofar, baseline=40), "6.4a Wrong: 'retention so far' by signup cohort. The last 13 cohorts are younger, so the end of the chart is always a fake rise", ax=axes[0], show_mr=False)
plot_xmr(xmr(r90, baseline=40), "6.4b Right: 90-day retention, only cohorts that have reached 90 days. Real drop at cohort 50, seen 13 weeks late", ax=axes[1], show_mr=False)
plot_xmr(xmr(r14, baseline=40), "6.4c Leading proxy: 14-day retention at a fixed age; available 11 weeks sooner, smaller effect size", ax=axes[2], show_mr=False)
for a in axes: a.axvline(coh_idx[50], color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()

*Deployment:* the table is `(cohort_week, age_days, retained)`; the chart is a filter on `age_days = 90`, not the latest value per cohort. Put the lag in the chart title ("latest mature cohort: 2025-xx-xx").
*Reading:* Wheeler, "Rational Sampling", Quality Digest, July 2015 (what you measure and when decides what the chart can see); Perla, Provost & Murray (2011), "The run chart: a simple analytical tool for learning from variation in healthcare processes", *BMJ Quality & Safety* 20(1), 46–51: https://qualitysafety.bmj.com/content/20/1/46 (practical guidance on charting outcomes that arrive late).

### 6.5 Mix shift and Simpson's paradox
Two acquisition channels: A converts at 10%, B at 3%, both perfectly stable. Between weeks 20 and 50 the share of traffic from B rises from 30% to 60%. The blended rate falls from 7.9% to 5.8% and the aggregate chart screams, yet nothing in either channel changed. The aggregate is right that *something* changed (the mix) and wrong about what to do about it. Component charts (6.5b) show stability; a mix/rate decomposition (printed) attributes the whole change to mix. The reverse also happens: a real deterioration in both channels can be hidden by a favourable mix shift.

In [ ]:
nw = 60; wk65 = pd.date_range("2025-01-05", periods=nw, freq="W-SUN")
share_B = np.clip(0.30 + (np.arange(nw) - 20) / 30 * 0.30, 0.30, 0.60)
visits = 4000
vA, vB = np.round(visits * (1 - share_B)), np.round(visits * share_B)
cA, cB = rng.binomial(vA.astype(int), 0.10), rng.binomial(vB.astype(int), 0.03)
rA, rB, rT = cA / vA, cB / vB, (cA + cB) / visits

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
plot_xmr(xmr(pd.Series(rT * 100, index=wk65), baseline=20, name="blended rate %"), "6.5a Aggregate conversion rate: a loud, real-looking decline", ax=axes[0], show_mr=False)
plot_xmr(xmr(pd.Series(rA * 100, index=wk65), baseline=20, name="channel A %"), "6.5b Channel A: stable", ax=axes[1], show_mr=False)
plot_xmr(xmr(pd.Series(rB * 100, index=wk65), baseline=20, name="channel B %"), "6.5c Channel B: stable. The aggregate moved because the mix moved", ax=axes[2], show_mr=False)
plt.tight_layout()

# Mix vs rate decomposition between the first 20 weeks and the last 10
def blend(sh, ra, rb): return (1 - sh) * ra + sh * rb
s0, s1 = share_B[:20].mean(), share_B[-10:].mean()
a0, a1, b0, b1 = rA[:20].mean(), rA[-10:].mean(), rB[:20].mean(), rB[-10:].mean()
total_change = blend(s1, a1, b1) - blend(s0, a0, b0)
mix_effect = blend(s1, a0, b0) - blend(s0, a0, b0)              # change the mix, hold rates
rate_effect = total_change - mix_effect
print(f"blended rate change: {100*total_change:+.2f} pts = mix effect {100*mix_effect:+.2f} pts + within-channel rate effect {100*rate_effect:+.2f} pts")

*Deployment:* every rate on a review should carry its mix dimensions in the registry; when the aggregate signals, the first query is the decomposition above, before anyone is asked to explain.
*Reading:* Simpson's paradox: https://en.wikipedia.org/wiki/Simpson%27s_paradox; Wheeler, "The Four Questions of Data Analysis", Quality Digest, Dec 2009 (homogeneity is the first question; a mixture is non-homogeneous by construction): https://www.qualitydigest.com/node/9504/

### 6.6 Roll-up masking: signals that vanish when units are aggregated
Twenty-five similar locations with 5% weekly noise each. One drops 20% at week 30. On its own chart that is a four-sigma Rule 1 signal in the first week. In the enterprise total it is a 0.8% change against total noise of about 1% (5% ÷ √25), so the total chart has to wait for a run to build, and a single noisy week can delay it indefinitely. Aggregation averages noise down by √N but averages a local signal down by N. The scalable answer is not twenty charts on a slide but a **signal grid**: run the chart per unit and show only the flags.

In [ ]:
N_LOC, nw = 25, 60
wk66 = pd.date_range("2025-01-05", periods=nw, freq="W-SUN")
loc = pd.DataFrame({f"S{i+1:02d}": np.round(1000 * rng.lognormal(0, 0.05, nw)) for i in range(N_LOC)}, index=wk66)
loc["S07"] = np.round(loc["S07"] * np.where(np.arange(nw) >= 30, 0.80, 1.0))
tot = loc.sum(axis=1)

res_tot = xmr(tot, baseline=30, name="all 25 locations")
res_s07 = xmr(loc["S07"], baseline=30, name="S07")
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
plot_xmr(res_tot, "6.6a Enterprise total: the 20% drop at one site is a 0.8% change in the total", ax=axes[0], show_mr=False)
plot_xmr(res_s07, "6.6b Site S07 on its own chart: unmistakable in week 1", ax=axes[1], show_mr=False)
for a in axes: a.axvline(wk66[30], color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()
# One draw proves nothing either way, so repeat the experiment 200 times and compare detection delays.
def first_delay(series, change_at=30, baseline=30):
    d = xmr(series, baseline=baseline).df
    hit = np.flatnonzero(d["signal"].values[change_at:])
    return int(hit[0]) if len(hit) else None

mc = []
for rep_i in range(200):
    sim = 1000 * rng.lognormal(0, 0.05, (nw, N_LOC))
    null_total = pd.Series(sim.sum(axis=1))                       # control: the same total with NO drop anywhere
    sim[30:, 6] *= 0.80
    mc.append(dict(total=first_delay(pd.Series(sim.sum(axis=1))), site=first_delay(pd.Series(sim[:, 6])),
                   total_no_change=first_delay(null_total)))
mc = pd.DataFrame(mc)
def summarise(col):
    v = mc[col]; found = v.dropna()
    return (f"flagged within 30 weeks in {100 * len(found) / len(v):3.0f}% of runs; median delay {found.median():.0f} weeks; "
            f"flagged in the first week in {100 * (v == 0).mean():3.0f}% of runs")
print("site S07 on its own chart      :", summarise("site"))
print("enterprise total (with drop)   :", summarise("total"))
print("enterprise total (NO change)   :", summarise("total_no_change"), "  <- false-alarm base rate of 4 rules on 30 noise points")
print("Read the total's flags against its no-change row: most of them are the ordinary false-alarm rate of the zone rules, not detection.")

def signal_grid(frame, baseline=None, rules=("r1", "r2", "r3", "r4"), title="Signal grid"):
    """One row per unit, one column per period; cell = strongest rule that fired (1 is strongest, 0 none)."""
    rank = {"r1": 4, "r4": 3, "r3": 2, "r2": 1}
    grid = pd.DataFrame(0, index=frame.columns, columns=frame.index)
    for c in frame.columns:
        d = xmr(frame[c], baseline=baseline, rules=rules).df
        for r, v in rank.items():
            grid.loc[c, d.index[d[r]]] = np.maximum(grid.loc[c, d.index[d[r]]], v)
    fig, ax = plt.subplots(figsize=(11, 0.28 * len(frame.columns) + 1.2))
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(["#f4f3ef", PAL["r2"], PAL["r3"], PAL["r4"], PAL["r1"]])
    ax.imshow(grid.values, aspect="auto", cmap=cmap, vmin=0, vmax=4, interpolation="nearest")
    ax.set_yticks(range(len(frame.columns))); ax.set_yticklabels(frame.columns, fontsize=8)
    ticks = np.arange(0, len(frame.index), 8); ax.set_xticks(ticks); ax.set_xticklabels([frame.index[i].strftime("%Y-%m-%d") for i in ticks], fontsize=8)
    ax.grid(False); ax.set_title(title + "  (colour: Rule 2 violet, Rule 3 yellow, Rule 4 orange, Rule 1 red)", loc="left")
    return grid

_ = signal_grid(loc, baseline=30, title="6.6c Signal grid for 25 locations: the roll-up blurs S07, the grid does not")

*Deployment:* charts live at the lowest unit that has a stable process (site, channel, cohort); the review sees a grid of flags plus the few charts that flagged. Expect a background rate of Rule 3/4 flags from noise: with 20 units and four rules, a few flags a week is normal and the grid should show persistence (same unit, consecutive weeks) rather than single cells.
*Reading:* Wheeler, "Rational Subgrouping", Quality Digest, 2015: https://www.qualitydigest.com/node/26952/; Mark Graban, *Measures of Success*, ch. on "Charts for every level" (the book argues for charts at the level where work happens; summary at https://www.leanblog.org/2024/02/measures-of-success-book-summary/).

### 6.7 Calendar artifacts: unequal month lengths
Monthly revenue from a constant daily run-rate varies ±5% just because February has 28 days and March 31, and a 23-business-day month beats a 20-business-day one by 15%. Against 2% noise those are "signals" every quarter. Divide by **selling days** (business days net of holidays for a B2B business; calendar days for a consumer one) and chart revenue per selling day. The same logic applies to 4-4-5 fiscal calendars (the 5-week month is +25%) and to weekly data crossing a leap day.

In [ ]:
months = pd.period_range("2023-01", periods=36, freq="M")
m_start = months.to_timestamp(how="start"); m_end = (months + 1).to_timestamp(how="start")
selling_days = np.array([np.busday_count(s.date(), e.date()) for s, e in zip(m_start, m_end)])   # Mon-Fri, no holiday table in this demo
rate_per_day = 10_000 * np.where(np.arange(36) >= 28, 1.06, 1.0)                                 # +6% run-rate change from month 29
monthly_rev = pd.Series(np.round(rate_per_day * selling_days * rng.lognormal(0, 0.02, 36)), index=m_start, name="monthly revenue")
per_day = monthly_rev / selling_days

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
plot_xmr(xmr(monthly_rev, baseline=24), "6.7a Wrong: monthly revenue. Month length drives the pattern (Feb dips, 23-day months spike)", ax=axes[0], show_mr=False)
plot_xmr(xmr(per_day, baseline=24, name="revenue per selling day"), "6.7b Right: revenue per selling day. The +6% run-rate change from month 29 is now visible", ax=axes[1], show_mr=False)
for a in axes: a.axvline(m_start[28], color=PAL["muted"], ls=":", lw=0.8)
plt.tight_layout()
print("selling days per month, first year:", selling_days[:12].tolist())

*Deployment:* keep a calendar dimension table with selling days per period per country; the "normalisation" field of the registry names it. Report both the normalised chart and the raw total (finance needs the total), but only the normalised one gets limits.
*Reading:* Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, 3rd ed., section 3.1 "Calendar adjustments": https://otexts.com/fpp3/transformations.html (unverified section link; the book is at https://otexts.com/fpp3/)

### 6.8 Targets vs natural limits: "expected range" and "goal" on one chart
A target is the voice of the customer or of finance; the limits are the voice of the process. They must be drawn in visibly different styles and named differently. Three situations, and the only sentence each one justifies:
1. **Goal inside the expected range.** The process will hit it some weeks and miss it others *purely by chance*. Celebrating or escalating week by week is reacting to noise (Deming's red beads). The right question is "what fraction of weeks will land above the goal if nothing changes?", and the only way to change that fraction is to change the process.
2. **Goal above the expected range.** The process as it stands cannot meet it. No amount of weekly pressure will; a change to the process is required, and the chart will show when it has happened.
3. **Goal below the expected range.** Already met every week; the goal adds nothing and a miss would itself be a signal.

In [ ]:
nw = 40; wk68 = pd.date_range("2025-01-05", periods=nw, freq="W-SUN")
perf = pd.Series(np.round(500 + rng.normal(0, 25, nw)), index=wk68, name="weekly jobs completed")
res_t = xmr(perf, name="weekly jobs completed")
from scipy import stats
fig, axes = plt.subplots(3, 1, figsize=(11, 9.5), sharex=True)
for ax, goal, lab in zip(axes, (510, 600, 420), ("6.8a Goal inside the expected range: hit by luck, missed by luck",
                                                 "6.8b Goal above the expected range: the current process cannot deliver it",
                                                 "6.8c Goal below the expected range: already reliably met")):
    plot_xmr(res_t, lab, ax=ax, show_mr=False, target=goal, target_label=f"goal = {goal}")
    ax.fill_between(wk68, res_t.lnpl, res_t.unpl, color=PAL["baseline"], alpha=0.25, linewidth=0)
    ax.text(wk68[0], res_t.unpl, " expected range (natural process limits)", fontsize=8, color=PAL["muted"], va="bottom")
    p_above = 1 - stats.norm.cdf((goal - res_t.cl) / res_t.sigma)       # rough, assumes near-normal noise within the limits
    print(f"goal {goal}: expected share of weeks at or above goal if nothing changes ≈ {100*p_above:.0f}%  (observed {100*(perf >= goal).mean():.0f}%)")
plt.tight_layout()

*Deployment:* the vocabulary on every chart and in every narration is "expected range" for the limits and "goal" for the target. Never colour weeks red for missing a goal that lies inside the expected range; colour the *goal line* instead, to show that it is the goal that is out of reach.
*Reading:* Stacey Barr, "Three Types of Useful KPI Targets" (targets on the central line, on a limit, or on predictability): https://www.staceybarr.com/?p=5337; Stacey Barr interview with Donald Wheeler on interpreting signals: https://www.staceybarr.com/?p=4527; Mark Graban, "Why Charts Beat Words for Understanding Variation and Performance": https://www.leanblog.org/2022/05/charts-understanding-variation-performance/; Deming, W. E. *Out of the Crisis*, MIT Press, 1986 (the red bead experiment).

## 7. Executive translation layer and the metric registry

### Three sentences
An executive does not need the rules; they need one of three statements, with the date and the size in business units:
1. **"Nothing changed."** The recent values are inside the expected range and show no sustained pattern. Variation is routine. No explanation is owed.
2. **"Something changed on [date] and has held for N periods, magnitude X."** A sustained shift: a run on one side of the central line (or a Rule 1 point followed by a run). Go and find the cause; if it is known and intended, start a new phase (Section 5).
3. **"One-off event on [date]."** A single value outside the expected range that the following values did not sustain. Explain the week; do not change the forecast.

`narrate()` below converts a chart into one of those. The logic is deliberately simple and conservative: the current regime is the run of points on one side of the central line ending at the latest point (one isolated week back across the line does not end it; two consecutive weeks do); it is a shift if it is at least `min_run` long and the run rule fired inside it, or if it contains a Rule 1 point and is at least `min_run_after_r1` long; a Rule 1 point in the last few periods that is not sustained is a one-off; everything else is noise. Once the rules say "shift", the *date* is estimated separately, by the least-squares change point (the split of the recent window into two means that explains the most variation), because "where the run started" drifts into the noise before the shift. A fourth, optional state, "watch", is returned when zone rules (3 or 4) are active but nothing is confirmed; whether the review shows it is a governance choice.

In [ ]:
def narrate(res, metric="the metric", units="", period="week", min_run=8, min_run_after_r1=3, watch=True, pct=True):
    """Translate an XmR result into one plain-English statement. Returns (state, sentence, details)."""
    d = res.df
    x, cl = d["x"].values, d["cl"].values
    n = len(d)
    side = np.sign(x - cl)
    # Current regime: walk back from the latest point while points stay on its side of the central line.
    # One isolated point back across the line does not end a regime; two consecutive ones do.
    s_last = side[n - 1]
    L, i = 0, n - 1
    while i >= 0 and s_last != 0:
        if side[i] == s_last:
            L = n - i; i -= 1
        elif i - 1 >= 0 and side[i - 1] == s_last:      # isolated crossing: bridge it
            i -= 1
        else:
            break
    run = d.iloc[n - L:] if L else d.iloc[0:0]
    r1_in_run = bool(run["r1"].any()) if L else False
    r2_in_run = bool(run["r2"].any()) if L else False
    # a Rule 1 event counts as the start of a shift only if the points AFTER it stayed on the same side
    n_after_r1 = (n - 1 - int(np.flatnonzero(run["r1"].values)[0] + (n - L))) if r1_in_run else -1
    start = run.index[0] if L else None
    fmt = (lambda v: f"{v:,.0f}{units}") if abs(cl[-1]) >= 100 else (lambda v: f"{v:,.2f}{units}")
    lo, hi = float(d["lnpl"].iloc[-1]), float(d["unpl"].iloc[-1])
    def _date(ix):
        return ix.date() if hasattr(ix, "date") else ix
    if (L >= min_run and r2_in_run) or (r1_in_run and n_after_r1 >= min_run_after_r1):
        # Date the shift with a least-squares change point inside the regime (plus a short margin before it):
        # the split that most reduces the squared error around two separate means. More reliable than
        # "where the run started", which drifts into pre-shift noise.
        lo = max(0, n - L - 8); best, best_cost = n - L, np.inf
        dev = x - cl
        for k in range(lo + 2, n - 1):
            a, b = dev[lo:k], dev[k:]
            cost = ((a - a.mean()) ** 2).sum() + ((b - b.mean()) ** 2).sum()
            if cost < best_cost:
                best, best_cost = k, cost
        run = d.iloc[best:]; L = len(run); start = run.index[0]
        mag = float(run["x"].mean() - run["cl"].mean())
        pct_s = f" ({100 * mag / run['cl'].mean():+.1f}%)" if pct and run["cl"].mean() != 0 else ""
        direction = "up" if mag > 0 else "down"
        sent = (f"Something changed in {metric} on {_date(start)}: it has been {direction} for {L} {period}s, "
                f"by about {fmt(abs(mag))}{pct_s} per {period} versus the previous level of {fmt(run['cl'].mean())}.")
        return "shift", sent, dict(start=start, periods=L, magnitude=mag)
    recent = d.iloc[-3:]
    if recent["r1"].any():
        ix = recent.index[recent["r1"]][-1]
        val = float(d.loc[ix, "x"])
        sent = (f"One-off event in {metric} on {_date(ix)}: {fmt(val)} against an expected range of {fmt(lo)} to {fmt(hi)}; "
                f"the {period}s since did not sustain it.")
        return "one_off", sent, dict(date=ix, value=val)
    if watch and (d.iloc[-4:][["r3", "r4"]].values.any()):
        sent = (f"Watch {metric}: recent {period}s lean to one side of the expected range (zone rule), but nothing is confirmed. "
                f"Expected range {fmt(lo)} to {fmt(hi)}.")
        return "watch", sent, {}
    sent = (f"Nothing changed in {metric}: the last {min(n, 8)} {period}s sit inside the expected range of {fmt(lo)} to {fmt(hi)} "
            f"around {fmt(cl[-1])}. The movement is routine variation; no explanation is needed.")
    return "noise", sent, {}


# Demonstrate on three synthetic situations plus the Section 5 phase chart
wk7 = pd.date_range("2025-01-05", periods=40, freq="W-SUN")
noise7 = pd.Series(np.round(800 + rng.normal(0, 30, 40)), index=wk7)
shift7 = pd.Series(np.round(800 + rng.normal(0, 30, 40) + np.where(np.arange(40) >= 28, 70, 0)), index=wk7)
spike7 = noise7.copy(); spike7.iloc[-2] = 1000
for label, s in [("stable series", noise7), ("sustained shift from week 29", shift7), ("spike in the penultimate week", spike7)]:
    r = xmr(s, baseline=24, rules=("r1", "r2"))            # Wheeler's default rule set; the registry decides per metric
    state, sentence, det = narrate(r, metric="weekly bookings", units="")
    print(f"[{label}] -> {state.upper()}\n   {sentence}\n")
state, sentence, det = narrate(res_ph, metric="weekly revenue", units="")
print(f"[Section 5 phase chart] -> {state.upper()}\n   {sentence}")

### The metric registry: how this becomes a product
Everything above is a *decision per metric*, and those decisions are what you productise. A registry row per metric captures them so that the chart, the narration and the review are generated, not hand-built. The fields that this notebook has shown to matter:

| Field | Values | Why it exists (section) |
|---|---|---|
| `grain` | day, week, month, event, cohort-age | the unit of logical comparability (0, 1, 6.4) |
| `seasonality_model` | none, dow-multiplicative, weekly-multiplicative, weekly-additive, holiday-aligned | which factors to apply and how (1, 2) |
| `maturity_stage` | new (cohort chart), provisional, standard | new units and graduation (3) |
| `trend_model` | none, log-linear (half-average), index-relative | growth handling (4) |
| `denominator` | none, column name, selling-days table, market index | rates, calendar, index charts (6.1, 6.7, 4) |
| `transformation` | none, log, logit, complement, time-between | bounded and rare metrics (6.2, 6.3) |
| `min_denominator` | number | pooling rule (6.1) |
| `rules` | r1 only, r1+r2, all four | sensitivity vs false alarms (0) |
| `baseline` | window spec + phase log reference | limits and their provenance (5) |
| `recalc_policy` | manual-on-signal (default); never automatic | governance (5) |
| `components` | list of additive child metrics; mix dimensions | localisation and Simpson's paradox (4, 6.5) |
| `owner`, `target`, `target_type` | person; number; central-line / limit / predictability | targets vs limits (6.8) |

A thin pipeline then reads the row, prepares the series (aggregate, deseasonalize, transform, index), builds the chart with the registered baseline, runs the registered rules, and narrates. Below is a minimal working version of that pipeline on three registry rows, reusing data from earlier sections.

In [ ]:
REGISTRY = [
    dict(metric="visits", grain="day", seasonality_model="dow-multiplicative", transformation="none", trend_model="none",
         baseline="first 18 weeks", rules=("r1", "r2", "r3", "r4"), recalc_policy="manual-on-signal", owner="ops"),
    dict(metric="orders", grain="week", seasonality_model="weekly-multiplicative", transformation="none", trend_model="none",
         baseline="first 104 weeks", rules=("r1", "r2"), recalc_policy="manual-on-signal", owner="commercial"),
    dict(metric="our weekly orders", grain="week", seasonality_model="none", transformation="none", trend_model="log-linear",
         baseline="first 52 weeks", rules=("r1", "r2", "r3", "r4"), recalc_policy="manual-on-signal", owner="growth"),
]
SOURCES = {"visits": daily, "orders": vol, "our weekly orders": ours}         # in production: SQL queries keyed by metric


def prepare(entry, series):
    """Apply the registry's comparability fixes to a raw series."""
    s = series.copy()
    if entry["seasonality_model"] == "dow-multiplicative":
        base = s.index < s.index[18 * 7]
        f = seasonal_factors(s, s.index.dayofweek, baseline_mask=base)
        s = deseasonalize(s, s.index.dayofweek, f)
    elif entry["seasonality_model"] == "weekly-multiplicative":
        p = np.arange(len(s)) % 52
        f = smooth_circular(seasonal_factors(s, p, baseline_mask=np.arange(len(s)) < 104), 5)
        s = deseasonalize(s, p, f)
    if entry["transformation"] == "log":
        s = np.log(s)
    return s


def baseline_points(entry):
    n_units = int(entry["baseline"].split()[1])
    return n_units * 7 if entry["grain"] == "day" else n_units


def run_registry(entry):
    s = prepare(entry, SOURCES[entry["metric"]])
    b = baseline_points(entry)
    if entry["trend_model"] == "log-linear":
        res = xmr_trend(s, baseline=b, log=True, rules=entry["rules"], name=entry["metric"])
    else:
        res = xmr(s, baseline=b, rules=entry["rules"], name=entry["metric"])
    state, sentence, det = narrate(res, metric=entry["metric"], period=entry["grain"])
    return res, state, sentence


for entry in REGISTRY:
    res, state, sentence = run_registry(entry)
    print(f"{entry['metric']:18s} [{entry['seasonality_model']}, trend={entry['trend_model']}, rules={'+'.join(entry['rules'])}] -> {state.upper()}")
    print("   " + sentence + "\n")

### Deployment notes (productising)
- **The registry is the product.** Charts and sentences are derived artefacts; version the registry, review changes to it like code, and log every phase decision against the metric it concerns.
- **Narration is conservative by design.** The three sentences map to Rule 1 and Rule 2; the zone rules only feed the optional "watch" state. If a review wants more sensitivity, change the registry's `rules`, not the sentence logic.
- **Do not auto-recalculate.** The pipeline reads the baseline from the registry; a human changes it (Section 5).
- **Sanity checks belong in the pipeline:** incomplete current period excluded, denominator floor applied, cohort maturity filter applied, holiday weeks annotated. Each is one of the failure modes above, encoded once.
- **Test with synthetic injections.** This notebook's generators (known shift at known date) are also the regression tests for the pipeline: the narration must say "shift on [that date]" and nothing else.

**Further reading (Section 7)**
- Cedric Chin, "Introducing Xmrit" (a free browser tool for XmR charts; useful to see how a product frames the same decisions): https://commoncog.com/introducing-xmrit/ and the Xmrit manual: https://xmrit.com/manual/
- Cedric Chin, "Executing on Becoming Data Driven: The Technicals", Commoncog: https://commoncog.com/executing-data-driven-the-technicals/
- Stacey Barr, "How to Explain the Value of XmR Charts for KPIs": https://staceybarr.com/?p=31025
- Mark Graban, "There's No Special Cause of Common Cause Variation" (the discipline of not explaining noise): https://www.leanblog.org/2024/04/theres-no-special-cause-of-common-cause-variation-tommy-john-surgeries

## 8. Reading list

All links were checked against a search index on 2026-11-08; items marked (unverified) could not be confirmed and the book/author reference stands on its own. Quality Digest articles are free to read with a registration.

### Theory: why the method is valid
- **Shewhart, W. A. (1931). *Economic Control of Quality of Manufactured Product*.** The origin. Best for: understanding that three-sigma limits are an economic choice, not a probability statement.
- **Wheeler, D. J. & Chambers, D. S. (2010). *Understanding Statistical Process Control*, 3rd ed., SPC Press.** The reference text. Best for: the constants, the "done right" XmR computation (ch. 9), rational subgrouping.
- **Wheeler, D. J. (2000). *Understanding Variation: The Key to Managing Chaos*, 2nd ed., SPC Press.** Best for: the shortest complete explanation of the chart and the four detection rules for managers.
- **Wheeler, D. J. (2003). *Making Sense of Data*, SPC Press.** Best for: service and administrative data; seasonal factors; the half-average trending chart (ch. 12).
- **Wheeler, "Why Three-Sigma Limits?"**, Quality Digest (1996): https://www.qualitydigest.com/aug/wheeler.html. Best for: the one-page answer to "why not two sigma / why not a p-value".
- **Wheeler, "Foundations of Shewhart's Charts"**, Quality Digest, Oct 1996: https://www.qualitydigest.com/oct96/spctool.html. Best for: why normality is not required.
- **Wheeler, "What Makes the XmR Chart Work?"**, Quality Digest, Nov 2012: https://www.qualitydigest.com/node/22342/. Best for: the two principles (comparability, local variation) this whole notebook is built on.
- **Wheeler, "Individual Charts Done Right and Done Wrong"**, Quality Digest, Feb 2010: https://www.qualitydigest.com/node/10006/. Best for: why the global standard deviation must not be used for limits.
- **Wheeler & Stauffer, "When Should We Use Extra Detection Rules?"**, Quality Digest: https://www.qualitydigest.com/node/30664. Best for: choosing the rule set per metric.
- **Wheeler, "Autocorrelated Data"**, Quality Digest, Aug 2017: https://www.qualitydigest.com/inside/statistics-column/autocorrelated-data-080717.html. Best for: what rolling averages and cumulatives do to the limits.
- **Wheeler, "The Four Questions of Data Analysis"**, Quality Digest, Dec 2009: https://www.qualitydigest.com/node/9504/. Best for: homogeneity as the first question (mix, roll-ups).
- **Wheeler, "Avoiding Statistical Jabberwocky"**, Quality Digest, Oct 2009: https://www.qualitydigest.com/node/9126/. Best for: the case against routine data transformation (read before using the logit).
- **Wheeler, "Hazardous Ideas About Charts for Individual Values"**, Quality Digest, 2026: https://www.qualitydigest.com/inside/statistics-article/hazardous-ideas-about-charts-individual-values-072726.html. Best for: current misconceptions about normality and false alarms.
- **Laney, D. B. (2002). "Improved Control Charts for Attributes", *Quality Engineering* 14(4), 531–537.** https://doi.org/10.1081/QEN-120003555. Best for: the p′ chart derivation. His short version: "P-Charts and U-Charts Work (But Only Sometimes)", Quality Digest, Feb 2007: https://www.qualitydigest.com/feb07/departments/what_works.shtml
- **Wheeler, "What About p-Charts?"**, Quality Digest, Sept 2011: https://www.qualitydigest.com/node/18604/. Best for: why XmR on the rate is the safe default.
- **Wheeler, "Charts for Rare Events"** (Dec 1996): https://www.qualitydigest.com/dec96/toolkit.html and **"Working with Rare Events"** (Oct 2011): https://www.qualitydigest.com/node/18838/. Best for: time-between and rate charts for low counts.
- **Cleveland et al. (1990). "STL: A Seasonal-Trend Decomposition Procedure Based on Loess", *Journal of Official Statistics* 6(1), 3–73.** Best for: what STL actually fits, so you know what it will absorb.
- **Benneyan, Lloyd & Plsek (2003). "Statistical process control as a tool for research and healthcare improvement", *Quality & Safety in Health Care* 12, 458–464.** https://qualitysafety.bmj.com/content/12/6/458. Best for: a peer-reviewed overview with non-manufacturing examples.
- **Western Electric rules**: https://en.wikipedia.org/wiki/Western_Electric_rules. Best for: the 4-of-5 variant and the history of zone tests.

### Implementation: how to deploy it
- **Graban, M. (2018). *Measures of Success: React Less, Lead Better, Improve More*.** Best for: the manager-facing argument and vocabulary; chapter summary at https://www.leanblog.org/2024/02/measures-of-success-book-summary/; worked sports examples at https://www.leanblog.org/tag/process-behavior-charts/
- **Stacey Barr, "How to Build an XmR Chart for Your KPI"**: https://staceybarr.com/?p=6848. Best for: a step-by-step build with the 2.66 constant and the 5-point minimum.
- **Stacey Barr, "3 Common Mistakes" case study**, parts 1–3: https://staceybarr.com/?p=2759, https://staceybarr.com/?p=2785, https://staceybarr.com/?p=2795. Best for: seasonality, measure choice, and signal rules in a KPI setting.
- **Stacey Barr, "Interpreting Signals in Trending KPIs"**: https://www.staceybarr.com/?p=19204. Best for: a practitioner's trending chart recipe.
- **Stacey Barr, "3 Steps to Set a Performance Measure Baseline"**: https://staceybarr.com/?p=20405. Best for: recalculation discipline.
- **Stacey Barr, "Three Types of Useful KPI Targets"**: https://www.staceybarr.com/?p=5337. Best for: targets vs limits vocabulary.
- **Stacey Barr, interview with Donald Wheeler**: https://www.staceybarr.com/?p=4527. Best for: Wheeler on KPIs in his own words.
- **Cedric Chin, "Process Behaviour Charts: More Than You Need To Know"**, Commoncog: https://commoncog.com/process-behaviour-charts-more-than-you-need/. Best for: the longest free practitioner treatment, with business examples.
- **Cedric Chin, "Becoming Data Driven, From First Principles"**: https://commoncog.com/becoming-data-driven-first-principles/ and **"Executing on Becoming Data Driven: The Technicals"**: https://commoncog.com/executing-data-driven-the-technicals/. Best for: the weekly business review as the home of these charts.
- **Xmrit** (free browser XmR tool): https://xmrit.com/about/ and manual https://xmrit.com/manual/. Best for: quick charts and for seeing a productised set of the same decisions.
- **statsmodels `STL`**: https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.STL.html. Best for: parameters (`seasonal`, `trend`, `robust`) that decide what gets absorbed.
- **Minitab, "Overview for Laney P′ Chart"**: https://support.minitab.com/minitab/help-and-how-to/quality-and-process-improvement/control-charts/how-to/attributes-charts/laney-p-chart/before-you-start/overview. Best for: a vendor's plain description and a diagnostic for overdispersion.
- **BPI Consulting (SPC for Excel), "Two Questions I Asked Dr. Wheeler"**: https://spcforexcel.com/knowledge/control-chart-basics/two-questions-i-asked-dr-wheeler/. Best for: Wheeler's own advice to keep extra rules "in your hip pocket".
- **NHS-R `NHSRplotthedots` vignette, "Number of points required"**: https://cloud.r-project.org/web/packages/NHSRplotthedots/vignettes/number-of-points-required.html. Best for: a practical discussion of trial limits from few points.
- **Provost, L. P. & Murray, S. K. (2022). *The Health Care Data Guide*, 2nd ed., Jossey-Bass.** Best for: T charts, G charts, rare events and lagged outcomes in a service setting.
- **Perla, Provost & Murray (2011). "The run chart", *BMJ Quality & Safety* 20(1), 46–51.** https://qualitysafety.bmj.com/content/20/1/46. Best for: what to do when you have too few points even for provisional limits.
- **Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, 3rd ed.**: https://otexts.com/fpp3/. Best for: calendar adjustments and STL from the forecasting side (and for recognising when you have left monitoring for forecasting).